# A1 multiscale ablation and official test

Attach only the prepared **multiscale `cat`** Kaggle dataset. The already obtained A1 alpha=0.25 official-test predictions and training OOF scores are embedded as a small checksummed bundle; this notebook does **not** run A1 test inference or need its checkpoints or uniscale feature bags. Select **GPU T4 ×2** and **Save Version → Save & Run All**. The notebook trains fixed and adaptive multiscale A1 concurrently (15 fold/seed fits each), verifies both, then evaluates those two on the 129 official test slides and compares them with the archived A1 test results. All outputs are written to `/kaggle/working/multiscale-a1/`.

The earlier project has already viewed CAMELYON16 official-test results; this comparison is exploratory on the existing holdout. The test data is not used for fitting or tuning this study.


In [ ]:
%%writefile /kaggle/working/baseline_dsmil.py
"""DSMIL baseline on prepared CAMELYON16 uniscale bags.

The model and optimizer follow dsmil.py/train_tcga.py in the repository.
The official test slides are only opened by --mode test, after CV finishes.
"""

from __future__ import annotations

import argparse
import copy
import csv
import hashlib
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, roc_auc_score, roc_curve
from torch import nn
from torch.nn import functional as F


class FCLayer(nn.Module):
    def __init__(self, in_size: int, out_size: int = 1):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(in_size, out_size))

    def forward(self, feats):
        return feats, self.fc(feats)


class BClassifier(nn.Module):
    def __init__(self, input_size: int, output_class: int = 1, anchors: int = 1):
        super().__init__()
        if output_class != 1 or anchors < 1:
            raise ValueError("This binary CAMELYON16 runner requires one class and at least one anchor")
        self.anchors = anchors
        self.q = nn.Sequential(nn.Linear(input_size, 128), nn.ReLU(),
                               nn.Linear(128, 128), nn.Tanh())
        self.v = nn.Identity()
        self.fcc = nn.Conv1d(output_class, output_class, kernel_size=input_size)

    def forward(self, feats, c):
        v = self.v(feats)
        q = self.q(feats).view(feats.shape[0], -1)
        if self.anchors == 1:
            # Exact released DSMIL critical-instance path.
            _, indices = torch.sort(c, 0, descending=True)
            critical = torch.index_select(feats, dim=0, index=indices[0, :])
        else:
            k = min(self.anchors, len(feats))
            indices = torch.topk(c[:, 0], k=k, sorted=True).indices
            critical = torch.index_select(feats, dim=0, index=indices)
        q_max = self.q(critical)
        attention = torch.mm(q, q_max.transpose(0, 1))
        attention = F.softmax(attention / torch.sqrt(torch.tensor(q.shape[1], dtype=torch.float32, device=feats.device)), 0)
        anchor_bags = torch.mm(attention.transpose(0, 1), v)
        bag = anchor_bags.mean(dim=0, keepdim=True).view(1, c.shape[1], v.shape[1])
        prediction = self.fcc(bag).view(1, -1)
        return prediction, attention, bag


class MILNet(nn.Module):
    def __init__(self, feature_dim: int = 512, anchors: int = 1):
        super().__init__()
        self.i_classifier = FCLayer(feature_dim, 1)
        self.b_classifier = BClassifier(feature_dim, 1, anchors=anchors)

    def forward(self, x):
        feats, classes = self.i_classifier(x)
        prediction, attention, bag = self.b_classifier(feats, classes)
        return classes, prediction, attention, bag


def make_model(device: torch.device, lr: float, weight_decay: float, epochs: int, anchors: int = 1):
    model = MILNet(anchors=anchors).to(device)
    for layer in model.modules():
        if isinstance(layer, (nn.Linear, nn.Conv1d)):
            nn.init.orthogonal_(layer.weight)
            if layer.bias is not None:
                nn.init.constant_(layer.bias, 0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.5, 0.9),
                                 weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs, 0.000005)
    return model, optimizer, scheduler


def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def read_csv(path: Path) -> list[dict]:
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))


def write_csv(path: Path, rows: list[dict], fields: list[str]) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    os.replace(tmp, path)


def write_json(path: Path, value: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")
    os.replace(tmp, path)


def load_manifest(prepared: Path) -> tuple[list[dict], dict]:
    completion = prepared / "PREPROCESSING_COMPLETE.json"
    if not completion.is_file():
        raise FileNotFoundError(f"Prepared dataset is incomplete: {completion}")
    info = json.loads(completion.read_text())
    if info.get("source") != "uniscale" or info.get("feature_dim") != 512 or info.get("seed") != 42:
        raise ValueError(f"Unexpected prepared source or split seed: {info}")
    rows = read_csv(prepared / "slides.csv")
    if len(rows) != 399 or len({r["slide_id"] for r in rows}) != 399:
        raise ValueError("Expected 399 unique slide rows")
    train = [r for r in rows if r["official_split"] == "train"]
    test = [r for r in rows if r["official_split"] == "test"]
    if len(train) != 270 or len(test) != 129:
        raise ValueError("Expected 270 official train and 129 official test slides")
    for row in rows:
        sid = row["slide_id"]
        row["label"] = int(row["label"])
        row["n_patches"] = int(row["n_patches"])
        if row["label"] not in (0, 1) or int(row["feature_dim"]) != 512:
            raise ValueError(f"Invalid slide label or dimension: {sid}")
        if Path(row["bag_path"]).is_absolute() or ".." in Path(row["bag_path"]).parts:
            raise ValueError(f"Unsafe bag path: {sid}")
        if not (prepared / row["bag_path"]).is_file():
            raise FileNotFoundError(f"Missing bag: {sid}")
        if row["official_split"] == "train":
            row["validation_fold"] = int(row["validation_fold"])
            if not 0 <= row["validation_fold"] < 5 or sid.startswith("test_"):
                raise ValueError(f"Invalid train fold: {sid}")
        elif row["validation_fold"] or not sid.startswith("test_"):
            raise ValueError(f"Invalid held-out slide: {sid}")
    if any(sum(r["validation_fold"] == fold for r in train) != 54 for fold in range(5)):
        raise ValueError("Expected 54 validation slides per fold")
    if next(r["label"] for r in rows if r["slide_id"] == "test_114") != 0:
        raise ValueError("test_114 must be normal per the chosen study protocol")
    return rows, info


def load_bag(prepared: Path, row: dict, device: torch.device) -> torch.Tensor:
    array = np.load(prepared / row["bag_path"], mmap_mode="r", allow_pickle=False)
    if array.shape != (row["n_patches"], 512) or array.dtype != np.dtype("float32"):
        raise ValueError(f"Invalid bag shape/dtype: {row['slide_id']}")
    # torch.tensor copies from the read-only memmap and transfers one bag at a time.
    return torch.tensor(array, dtype=torch.float32, device=device)


def bag_step(model: MILNet, feats: torch.Tensor, label: int, criterion):
    instance_logits, bag_logits, attention, _ = model(feats)
    max_logit = instance_logits.max(dim=0).values.view(1, 1)
    target = torch.tensor([[float(label)]], device=feats.device)
    loss = 0.5 * criterion(bag_logits.view(1, 1), target) + 0.5 * criterion(max_logit, target)
    probability = torch.sigmoid(bag_logits).item()
    return loss, probability, instance_logits, attention


def attention_overlap(attention: torch.Tensor) -> torch.Tensor:
    """Mean off-diagonal cosine between anchor attention distributions."""
    k = attention.shape[1]
    if k < 2:
        return attention.new_zeros(())
    normalized = F.normalize(attention, p=2, dim=0)
    similarity = normalized.transpose(0, 1) @ normalized
    off_diagonal = ~torch.eye(k, dtype=torch.bool, device=attention.device)
    return similarity[off_diagonal].mean()


def train_epoch(model, rows, prepared, device, optimizer, criterion, seed, diversity_weight):
    model.train()
    order = np.random.default_rng(seed).permutation(len(rows))
    classification_total = 0.0
    objective_total = 0.0
    overlap_total = 0.0
    for index in order:
        row = rows[int(index)]
        feats = load_bag(prepared, row, device)
        feats = feats[torch.randperm(len(feats), device=device)]
        optimizer.zero_grad(set_to_none=True)
        classification_loss, _, _, attention = bag_step(model, feats, row["label"], criterion)
        overlap = attention_overlap(attention) if diversity_weight > 0 else attention.new_zeros(())
        objective = classification_loss + diversity_weight * overlap
        objective.backward()
        optimizer.step()
        classification_total += float(classification_loss.detach().cpu())
        objective_total += float(objective.detach().cpu())
        overlap_total += float(overlap.detach().cpu())
    return (classification_total / len(rows), objective_total / len(rows),
            overlap_total / len(rows))


@torch.no_grad()
def predict(model, rows, prepared, device, criterion):
    model.eval()
    predictions = []
    diagnostics = []
    total_loss = 0.0
    for row in rows:
        feats = load_bag(prepared, row, device)
        loss, probability, instance_logits, attention = bag_step(model, feats, row["label"], criterion)
        total_loss += float(loss.detach().cpu())
        predictions.append({"slide_id": row["slide_id"], "label": row["label"],
                            "probability": probability})
        if model.b_classifier.anchors > 1:
            k = min(model.b_classifier.anchors, len(feats))
            indices = torch.topk(instance_logits[:, 0], k=k, sorted=True).indices
            normalized = F.normalize(feats[indices], dim=1)
            similarities = normalized @ normalized.T
            off_diagonal = ~torch.eye(k, dtype=torch.bool, device=device)
            cosine_mean = float(similarities[off_diagonal].mean().item()) if k > 1 else None
            entropy = -(attention * attention.clamp_min(1e-12).log()).sum(dim=0)
            entropy = entropy / np.log(max(len(feats), 2))
            diagnostics.append({"slide_id": row["slide_id"],
                                "anchor_row_indices": indices.cpu().tolist(),
                                "anchor_instance_logits": instance_logits[indices, 0].cpu().tolist(),
                                "mean_pairwise_anchor_cosine": cosine_mean,
                                "mean_pairwise_attention_cosine": float(attention_overlap(attention).item()),
                                "normalized_attention_entropy": entropy.cpu().tolist()})
    return total_loss / len(rows), predictions, diagnostics


def auc(rows: list[dict]) -> float:
    return float(roc_auc_score([r["label"] for r in rows], [r["probability"] for r in rows]))


def threshold_from_oof(rows: list[dict]) -> float:
    labels = [r["label"] for r in rows]
    scores = [r["probability"] for r in rows]
    fpr, tpr, thresholds = roc_curve(labels, scores)
    candidates = np.flatnonzero(np.isfinite(thresholds))
    winner = candidates[np.argmax((tpr - fpr)[candidates])]
    return float(thresholds[winner])


def train_cv_single_seed(args, rows, info, device):
    prepared = args.prepared_root
    out = args.output_root
    if out.exists() and any(out.iterdir()):
        raise FileExistsError(f"Choose an empty output root for a new CV run: {out}")
    out.mkdir(parents=True)
    config = {"architecture": "DSMIL FCLayer+BClassifier" if args.anchors == 1 else "DSMIL mean-pooled Top-K anchor bags",
              "anchors": args.anchors,
              "diversity_weight": args.diversity_weight,
              "diversity_penalty": "mean pairwise cosine of L2-normalized anchor attention columns",
              "feature_source": "uniscale", "feature_dim": 512, "seed": args.seed,
              "epochs": args.epochs, "patience": args.patience, "lr": args.lr,
              "weight_decay": args.weight_decay, "device": str(device),
              "checkpoint_metric": "validation_auc", "decision_threshold": "Youden J on 270 OOF scores",
              "test_rule": "mean of five fold model probabilities; apply one OOF threshold",
              "prepared_input": str(prepared), "prepared_info": info,
              "manifest_sha256": hashlib.sha256((prepared / "slides.csv").read_bytes()).hexdigest()}
    write_json(out / "config.json", config)
    criterion = nn.BCEWithLogitsLoss()
    all_oof = []
    fold_summary = []
    for fold in range(5):
        train = [r for r in rows if r["official_split"] == "train" and r["validation_fold"] != fold]
        valid = [r for r in rows if r["official_split"] == "train" and r["validation_fold"] == fold]
        assert len(train) == 216 and len(valid) == 54
        seed_all(args.seed)
        model, optimizer, scheduler = make_model(device, args.lr, args.weight_decay, args.epochs, args.anchors)
        best_auc = -1.0
        best_loss = float("inf")
        best_epoch = 0
        stale = 0
        history = []
        fold_started = time.monotonic()
        for epoch in range(1, args.epochs + 1):
            started = time.monotonic()
            train_loss, train_objective, train_overlap = train_epoch(
                model, train, prepared, device, optimizer, criterion,
                args.seed + fold * 100000 + epoch, args.diversity_weight)
            valid_loss, predictions, diagnostics = predict(model, valid, prepared, device, criterion)
            valid_auc = auc(predictions)
            accuracy05 = float(accuracy_score([r["label"] for r in predictions],
                                                 [int(r["probability"] >= 0.5) for r in predictions]))
            row = {"fold": fold, "epoch": epoch, "train_loss": train_loss,
                   "train_objective": train_objective,
                   "train_attention_overlap": train_overlap,
                   "valid_loss": valid_loss, "valid_auc": valid_auc,
                   "valid_accuracy_0.5": accuracy05, "lr": optimizer.param_groups[0]["lr"],
                   "elapsed_seconds": round(time.monotonic() - started, 2)}
            history.append(row)
            improved = valid_auc > best_auc + 1e-12 or (abs(valid_auc - best_auc) <= 1e-12 and valid_loss < best_loss)
            if improved:
                best_auc, best_loss, best_epoch, stale = valid_auc, valid_loss, epoch, 0
                torch.save(model.state_dict(), out / f"fold_{fold}.pt")
                selected = [{"fold": fold, **r} for r in predictions]
                write_csv(out / f"fold_{fold}_oof.csv", selected,
                          ["fold", "slide_id", "label", "probability"])
                if args.anchors > 1:
                    write_json(out / f"fold_{fold}_anchor_diagnostics.json",
                               {"fold": fold, "epoch": epoch, "slides": diagnostics})
            else:
                stale += 1
            scheduler.step()
            print(f"fold={fold} epoch={epoch} train_loss={train_loss:.4f} overlap={train_overlap:.4f} val_loss={valid_loss:.4f} val_auc={valid_auc:.4f} best={best_auc:.4f} sec={row['elapsed_seconds']:.1f}", flush=True)
            if stale >= args.patience:
                break
        write_csv(out / f"fold_{fold}_history.csv", history, list(history[0]))
        selected = read_csv(out / f"fold_{fold}_oof.csv")
        for r in selected:
            r["fold"] = int(r["fold"])
            r["label"] = int(r["label"])
            r["probability"] = float(r["probability"])
        all_oof.extend(selected)
        model.load_state_dict(torch.load(out / f"fold_{fold}.pt", map_location=device, weights_only=True))
        selected_train_loss, train_predictions, _ = predict(model, train, prepared, device, criterion)
        train_accuracy = float(accuracy_score([r["label"] for r in train_predictions],
                                              [int(r["probability"] >= 0.5) for r in train_predictions]))
        valid_labels = [r["label"] for r in selected]
        valid_classes = [int(r["probability"] >= 0.5) for r in selected]
        tn, fp, fn, tp = confusion_matrix(valid_labels, valid_classes, labels=[0, 1]).ravel()
        fold_summary.append({"fold": fold, "best_epoch": best_epoch,
                             "validation_auc": best_auc, "validation_loss": best_loss,
                             "training_loss_at_best_checkpoint": selected_train_loss,
                             "training_accuracy_0.5": train_accuracy,
                             "validation_accuracy_0.5": float(accuracy_score(valid_labels, valid_classes)),
                             "validation_balanced_accuracy_0.5": float(balanced_accuracy_score(valid_labels, valid_classes)),
                             "validation_sensitivity_0.5": float(tp / (tp + fn)),
                             "validation_specificity_0.5": float(tn / (tn + fp)),
                             "elapsed_seconds": round(time.monotonic() - fold_started, 1)})
        write_json(out / f"fold_{fold}_summary.json", fold_summary[-1])
        del model, optimizer, scheduler
        if device.type == "cuda":
            torch.cuda.empty_cache()
    all_oof.sort(key=lambda r: r["slide_id"])
    if len(all_oof) != 270 or len({r["slide_id"] for r in all_oof}) != 270:
        raise ValueError("OOF predictions must cover the 270 train slides exactly once")
    write_csv(out / "oof_predictions.csv", all_oof, ["fold", "slide_id", "label", "probability"])
    threshold = threshold_from_oof(all_oof)
    summary = {"folds": fold_summary, "mean_validation_auc": float(np.mean([r["validation_auc"] for r in fold_summary])),
               "std_validation_auc": float(np.std([r["validation_auc"] for r in fold_summary], ddof=1)),
               "oof_auc": auc(all_oof), "oof_threshold": threshold,
               "oof_accuracy_at_threshold": float(accuracy_score([r["label"] for r in all_oof],
                                                                   [int(r["probability"] >= threshold) for r in all_oof])),
               "test_evaluated": False}
    write_json(out / "cv_summary.json", summary)
    print(json.dumps(summary, indent=2), flush=True)


def train_cv(args, rows, info, device):
    out = args.output_root
    if out.exists() and any(out.iterdir()):
        raise FileExistsError(f"Choose an empty output root: {out}")
    out.mkdir(parents=True, exist_ok=True)
    write_json(out / "config.json", {
        "seeds": args.seeds, "folds": 5, "anchors": args.anchors,
        "diversity_weight": args.diversity_weight,
        "manifest_sha256": hashlib.sha256((args.prepared_root / "slides.csv").read_bytes()).hexdigest(),
        "test_ensemble": "arithmetic mean of all 15 seed-by-fold checkpoint probabilities",
        "threshold": "Youden J on three-seed averaged OOF probabilities",
    })
    all_oof = []
    seed_means = []
    runs = []
    for seed in args.seeds:
        single = copy.copy(args)
        single.seed = seed
        single.output_root = out / f"seed_{seed}"
        train_cv_single_seed(single, rows, info, device)
        report = json.loads((single.output_root / "cv_summary.json").read_text())
        seed_means.append(report["mean_validation_auc"])
        runs.extend({"seed": seed, **fold} for fold in report["folds"])
        all_oof.extend({"seed": seed, **r} for r in read_csv(single.output_root / "oof_predictions.csv"))
    if len(runs) != 5 * len(args.seeds) or len(all_oof) != 270 * len(args.seeds):
        raise ValueError("Incomplete repeated CV")
    write_csv(out / "oof_predictions_all_seeds.csv", all_oof,
              ["seed", "fold", "slide_id", "label", "probability"])
    grouped = {}
    for r in all_oof:
        grouped.setdefault(r["slide_id"], []).append(r)
    if len(grouped) != 270 or any(len(v) != len(args.seeds) or
                                  len({x["fold"] for x in v}) != 1 or
                                  len({x["label"] for x in v}) != 1 for v in grouped.values()):
        raise ValueError("Each training slide needs one OOF prediction per seed on the same fold")
    ensemble_oof = [{"slide_id": sid, "fold": values[0]["fold"],
                     "label": int(values[0]["label"]),
                     "probability": float(np.mean([float(x["probability"]) for x in values]))}
                    for sid, values in sorted(grouped.items())]
    write_csv(out / "oof_predictions_ensemble.csv", ensemble_oof,
              ["slide_id", "fold", "label", "probability"])
    threshold = threshold_from_oof(ensemble_oof)
    summary = {
        "seeds": args.seeds, "runs": runs, "seed_mean_validation_auc": seed_means,
        "mean_validation_auc": float(np.mean(seed_means)),
        "std_seed_mean_validation_auc": float(np.std(seed_means, ddof=1)),
        "fold_auc_by_seed": {str(fold): [next(x["validation_auc"] for x in runs
                                                  if x["seed"] == seed and x["fold"] == fold)
                                        for seed in args.seeds] for fold in range(5)},
        "oof_ensemble_auc": auc(ensemble_oof), "oof_ensemble_threshold": threshold,
        "test_evaluated": False,
    }
    write_json(out / "cv_summary.json", summary)
    print(json.dumps(summary, indent=2), flush=True)


def evaluate_test(args, rows, device):
    prepared = args.prepared_root
    cv_root = args.cv_root
    summary = json.loads((cv_root / "cv_summary.json").read_text())
    config = json.loads((cv_root / "config.json").read_text())
    if len(summary["runs"]) != 5 * len(args.seeds) or config["seeds"] != args.seeds or config["manifest_sha256"] != hashlib.sha256((prepared / "slides.csv").read_bytes()).hexdigest() or config["anchors"] != args.anchors or config["diversity_weight"] != args.diversity_weight:
        raise ValueError("CV output and prepared manifest do not match")
    test = [r for r in rows if r["official_split"] == "test"]
    assert len(test) == 129
    out = args.output_root
    if out.exists() and any(out.iterdir()):
        raise FileExistsError(f"Choose an empty test output root: {out}")
    out.mkdir(parents=True)
    per_fold = []
    criterion = nn.BCEWithLogitsLoss()
    for seed in args.seeds:
        for fold in range(5):
            checkpoint = cv_root / f"seed_{seed}" / f"fold_{fold}.pt"
            model = MILNet(anchors=args.anchors).to(device)
            model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
            _, predictions, _ = predict(model, test, prepared, device, criterion)
            per_fold.extend({"seed": seed, "fold": fold, **r} for r in predictions)
            del model
            if device.type == "cuda":
                torch.cuda.empty_cache()
    if len(per_fold) != 129 * 5 * len(args.seeds):
        raise ValueError("Incomplete test predictions")
    write_csv(out / "test_predictions_per_checkpoint.csv", per_fold,
              ["seed", "fold", "slide_id", "label", "probability"])
    by_slide = {r["slide_id"]: [] for r in test}
    for r in per_fold:
        by_slide[r["slide_id"]].append(r["probability"])
    if any(len(v) != 5 * len(args.seeds) for v in by_slide.values()):
        raise ValueError("Each test slide must have one prediction from every checkpoint")
    threshold = float(summary["oof_ensemble_threshold"])
    ensemble = [{"slide_id": r["slide_id"], "label": r["label"],
                 "probability": float(np.mean(by_slide[r["slide_id"]])),
                 "prediction": int(np.mean(by_slide[r["slide_id"]]) >= threshold)} for r in test]
    write_csv(out / "test_predictions_ensemble.csv", ensemble,
              ["slide_id", "label", "probability", "prediction"])
    labels = [r["label"] for r in ensemble]
    predictions = [r["prediction"] for r in ensemble]
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    report = {"test_slides": 129, "ensemble": "mean of all seed-by-fold checkpoint probabilities",
              "n_models": 5 * len(args.seeds), "seeds": args.seeds,
              "threshold_from_oof": threshold, "test_auc": auc(ensemble),
              "test_accuracy": float(accuracy_score(labels, predictions)),
              "test_balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
              "test_sensitivity": float(tp / (tp + fn)),
              "test_specificity": float(tn / (tn + fp)),
              "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
              "test_114_label": 0, "cv_summary": str(cv_root)}
    write_json(out / "test_summary.json", report)
    print(json.dumps(report, indent=2), flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--mode", choices=["cv", "test"], default="cv")
    parser.add_argument("--prepared-root", type=Path, required=True)
    parser.add_argument("--output-root", type=Path, required=True)
    parser.add_argument("--cv-root", type=Path, help="Required for test mode")
    parser.add_argument("--seeds", nargs="+", type=int, default=[42, 43, 44])
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--patience", type=int, default=10)
    parser.add_argument("--lr", type=float, default=0.0001)
    parser.add_argument("--weight-decay", type=float, default=0.001)
    parser.add_argument("--anchors", type=int, default=1,
                        help="1 reproduces baseline DSMIL; 4 is the Phase 2 Top-K experiment")
    parser.add_argument("--diversity-weight", type=float, default=0.0,
                        help="Phase 3 uses 0.1; 0 leaves Phases 1 and 2 unchanged")
    args = parser.parse_args()
    if args.anchors < 1:
        parser.error("--anchors must be >= 1")
    if args.diversity_weight < 0 or (args.diversity_weight > 0 and args.anchors < 2):
        parser.error("Diversity requires a nonnegative weight and at least two anchors")
    if args.mode == "test" and args.cv_root is None:
        parser.error("--cv-root is required for test mode")
    if len(args.seeds) != 3 or len(set(args.seeds)) != 3:
        parser.error("Provide exactly three distinct training seeds")
    rows, info = load_manifest(args.prepared_root)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed_all(args.seeds[0])
    if args.mode == "cv":
        train_cv(args, rows, info, device)
    else:
        evaluate_test(args, rows, device)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/ab_validation_runner.py
"""Single-run, validation-only CAMELYON16 A/B experiments on prepared uniscale bags.

The Phase 1-3 runner is imported unchanged for data validation and shared
utilities. This module has no official-test evaluation path.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import math
import shutil
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import sklearn
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from torch import nn
from torch.nn import functional as F

import baseline_dsmil as base


BASELINE_SHA256 = "32dc2a316278566e4c41dbe4fc01ca04e003c726beffcc8b7311f59f13cd2385"
PREPARED_SHA256 = "12d13547a03ce93dacf2087fe237f16f9a69d1ac2611c6a6b88dadab945ca604"
SEEDS = (42, 43, 44)
FOLDS = tuple(range(5))
EPOCHS, PATIENCE = 50, 10
LR, WEIGHT_DECAY = 0.0001, 0.001


@dataclass(frozen=True)
class Variant:
    name: str
    anchors: int
    alpha: float = 0.0
    fusion: str = "mean"
    diversity_weight: float = 0.0


NEW_VARIANTS = (
    Variant("A1_alpha_025", 4, alpha=0.25),
    Variant("A1_alpha_050", 4, alpha=0.50),
    Variant("A1_alpha_075", 4, alpha=0.75),
    Variant("A2_learned_fusion", 4, fusion="learned"),
    Variant("B_K2_L0", 2),
    Variant("B_K2_L01", 2, diversity_weight=0.1),
)
PHASE2 = Variant("Phase2_K4_L0", 4)
PHASE3 = Variant("Phase3_K4_L01", 4, diversity_weight=0.1)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def select_diverse(feats: torch.Tensor, scores: torch.Tensor,
                   original_ids: torch.Tensor, alpha: float, anchors: int):
    """Return original-bag positions and rank records; selection is discrete."""
    if not 0.0 <= alpha <= 1.0 or len(feats) < 1:
        raise ValueError("Invalid alpha or empty bag")
    k = min(anchors, len(feats))
    if alpha == 0.0:
        indices = torch.topk(scores, k=k, sorted=True).indices
        ranks = [{"score_rank": rank + 1, "diversity_rank": None}
                 for rank in range(k)]
        return indices, ranks
    score_values = scores.detach().cpu().numpy()
    row_ids = original_ids.detach().cpu().numpy()
    ordering = np.lexsort((row_ids, -score_values))
    candidates = ordering[:min(32, len(feats))].tolist()
    chosen = [candidates[0]]
    rank_records = [{"score_rank": 1, "diversity_rank": None}]
    candidate_feats = F.normalize(feats.detach()[candidates], p=2, dim=1, eps=1e-12)
    cosine = (candidate_feats @ candidate_feats.T).cpu().numpy()
    candidate_pos = {index: position for position, index in enumerate(candidates)}
    while len(chosen) < k:
        remaining = [index for index in candidates if index not in chosen]
        score_ranks = {index: rank for rank, index in enumerate(remaining, 1)}
        distances = {index: 1.0 - float(max(cosine[candidate_pos[index], candidate_pos[j]]
                                            for j in chosen)) for index in remaining}
        diversity_order = sorted(remaining, key=lambda index: (-distances[index], int(row_ids[index])))
        diversity_ranks = {index: rank for rank, index in enumerate(diversity_order, 1)}
        winner = min(remaining, key=lambda index: (
            (1.0 - alpha) * score_ranks[index] + alpha * diversity_ranks[index],
            -float(score_values[index]), int(row_ids[index])))
        chosen.append(winner)
        rank_records.append({"score_rank": score_ranks[winner],
                             "diversity_rank": diversity_ranks[winner]})
    return torch.as_tensor(chosen, dtype=torch.long, device=feats.device), rank_records


class ABBClassifier(base.BClassifier):
    def __init__(self, variant: Variant):
        super().__init__(512, 1, anchors=variant.anchors)
        self.variant = variant
        self.fusion_u = nn.Parameter(torch.zeros(512)) if variant.fusion == "learned" else None

    def forward(self, feats, c, original_ids=None):
        if original_ids is None:
            original_ids = torch.arange(len(feats), device=feats.device)
        v = self.v(feats)
        q = self.q(feats).view(feats.shape[0], -1)
        indices, ranks = select_diverse(
            feats, c[:, 0], original_ids, self.variant.alpha, self.anchors)
        critical = torch.index_select(feats, dim=0, index=indices)
        q_max = self.q(critical)
        attention = torch.mm(q, q_max.transpose(0, 1))
        attention = F.softmax(
            attention / torch.sqrt(torch.tensor(q.shape[1], dtype=torch.float32, device=feats.device)), 0)
        anchor_bags = torch.mm(attention.transpose(0, 1), v)
        if self.variant.fusion == "learned":
            weights = F.softmax(anchor_bags @ self.fusion_u, dim=0)
            fused = torch.sum(weights[:, None] * anchor_bags, dim=0, keepdim=True)
        else:
            weights = torch.full((len(indices),), 1.0 / len(indices), device=feats.device)
            fused = anchor_bags.mean(dim=0, keepdim=True)
        bag = fused.view(1, c.shape[1], v.shape[1])
        prediction = self.fcc(bag).view(1, -1)
        details = {"indices": indices, "rank_records": ranks,
                   "anchor_bags": anchor_bags, "fusion_weights": weights}
        return prediction, attention, bag, details


class ABMILNet(nn.Module):
    def __init__(self, variant: Variant):
        super().__init__()
        self.i_classifier = base.FCLayer(512, 1)
        self.b_classifier = ABBClassifier(variant)

    def forward(self, x, original_ids=None):
        feats, classes = self.i_classifier(x)
        prediction, attention, bag, details = self.b_classifier(feats, classes, original_ids)
        return classes, prediction, attention, bag, details


def make_model(variant: Variant, device: torch.device):
    model = ABMILNet(variant).to(device)
    for layer in model.modules():
        if isinstance(layer, (nn.Linear, nn.Conv1d)):
            nn.init.orthogonal_(layer.weight)
            if layer.bias is not None:
                nn.init.constant_(layer.bias, 0)
    # fusion_u is initialized to zero in ABBClassifier and is not a Linear module.
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.5, 0.9),
                                 weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS, 0.000005)
    return model, optimizer, scheduler


def bag_step(model, feats, label, criterion, original_ids=None):
    instance_logits, bag_logits, attention, _, details = model(feats, original_ids)
    max_logit = instance_logits.max(dim=0).values.view(1, 1)
    target = torch.tensor([[float(label)]], device=feats.device)
    loss = 0.5 * criterion(bag_logits.view(1, 1), target) + 0.5 * criterion(max_logit, target)
    probability = torch.sigmoid(bag_logits).item()
    return loss, probability, instance_logits, attention, details


def train_epoch(model, train, prepared, device, optimizer, criterion, shuffle_seed, weight):
    model.train()
    order = np.random.default_rng(shuffle_seed).permutation(len(train))
    classification_total = objective_total = overlap_total = 0.0
    for index in order:
        row = train[int(index)]
        feats = base.load_bag(prepared, row, device)
        permutation = torch.randperm(len(feats), device=device)
        feats = feats[permutation]
        optimizer.zero_grad(set_to_none=True)
        loss, _, _, attention, _ = bag_step(model, feats, row["label"], criterion, permutation)
        overlap = base.attention_overlap(attention) if weight > 0 else attention.new_zeros(())
        objective = loss + weight * overlap
        if not torch.isfinite(objective):
            raise ValueError(f"Nonfinite training objective for {row['slide_id']}")
        objective.backward()
        optimizer.step()
        classification_total += float(loss.detach().cpu())
        objective_total += float(objective.detach().cpu())
        overlap_total += float(overlap.detach().cpu())
    count = len(train)
    return classification_total / count, objective_total / count, overlap_total / count


@torch.no_grad()
def predict(model, rows, prepared, device, criterion, diagnostics_enabled=True):
    model.eval()
    predictions, diagnostics = [], []
    total_loss = 0.0
    for row in rows:
        feats = base.load_bag(prepared, row, device)
        loss, probability, logits, attention, details = bag_step(model, feats, row["label"], criterion)
        total_loss += float(loss.detach().cpu())
        predictions.append({"slide_id": row["slide_id"], "label": row["label"],
                            "probability": probability})
        if not diagnostics_enabled:
            continue
        indices = details["indices"]
        k = len(indices)
        selected = F.normalize(feats[indices], p=2, dim=1, eps=1e-12)
        feature_cos = selected @ selected.T
        normalized_bags = F.normalize(details["anchor_bags"], p=2, dim=1, eps=1e-12)
        bag_cos = normalized_bags @ normalized_bags.T
        off = ~torch.eye(k, dtype=torch.bool, device=device)
        weights = details["fusion_weights"]
        diagnostics.append({
            "slide_id": row["slide_id"], "label": row["label"],
            "anchor_row_indices": indices.cpu().tolist(),
            "anchor_instance_logits": logits[indices, 0].cpu().tolist(),
            "anchor_rank_records": details["rank_records"],
            "mean_pairwise_anchor_cosine": float(feature_cos[off].mean().item()) if k > 1 else None,
            "mean_pairwise_attention_cosine": float(base.attention_overlap(attention).item()),
            "mean_pairwise_anchor_bag_cosine": float(bag_cos[off].mean().item()) if k > 1 else None,
            "fusion_weights": weights.cpu().tolist(),
            "fusion_weight_entropy": float((-(weights * weights.clamp_min(1e-12).log()).sum()).item()),
            "normalized_attention_entropy": (
                -(attention * attention.clamp_min(1e-12).log()).sum(dim=0) /
                np.log(max(len(feats), 2))).cpu().tolist(),
        })
    return total_loss / len(rows), predictions, diagnostics


def train_variant(variant: Variant, rows: list[dict], info: dict,
                  prepared: Path, root: Path, device: torch.device,
                  prepared_hash: str, runner_hash: str):
    out = root / variant.name
    if out.exists():
        raise FileExistsError(f"Output directory already exists: {out}")
    out.mkdir(parents=True)
    config = {"variant": variant.__dict__, "seeds": list(SEEDS), "folds": list(FOLDS),
              "feature_source": "uniscale", "feature_dim": 512,
              "epochs": EPOCHS, "patience": PATIENCE, "lr": LR, "weight_decay": WEIGHT_DECAY,
              "adam_betas": [0.5, 0.9], "scheduler": "CosineAnnealingLR",
              "eta_min": 0.000005, "checkpoint": "max validation AUC; tie by min validation loss",
              "probability": "sigmoid(bag logit)", "prepared_info": info,
              "prepared_manifest_sha256": prepared_hash, "runner_sha256": runner_hash,
              "baseline_sha256": BASELINE_SHA256, "device": str(device),
              "versions": {"torch": str(torch.__version__), "numpy": np.__version__,
                           "sklearn": sklearn.__version__}, "test_evaluated": False}
    base.write_json(out / "config.json", config)
    criterion = nn.BCEWithLogitsLoss()
    all_oof, runs, seed_means = [], [], []
    for seed in SEEDS:
        seed_root = out / f"seed_{seed}"
        seed_root.mkdir()
        seed_oof, fold_summaries = [], []
        for fold in FOLDS:
            train = [r for r in rows if r["official_split"] == "train" and r["validation_fold"] != fold]
            valid = [r for r in rows if r["official_split"] == "train" and r["validation_fold"] == fold]
            assert len(train) == 216 and len(valid) == 54
            base.seed_all(seed)
            model, optimizer, scheduler = make_model(variant, device)
            best_auc, best_loss, best_epoch, stale = -1.0, float("inf"), 0, 0
            history = []
            fold_start = time.monotonic()
            for epoch in range(1, EPOCHS + 1):
                epoch_start = time.monotonic()
                train_loss, train_objective, train_overlap = train_epoch(
                    model, train, prepared, device, optimizer, criterion,
                    seed + fold * 100000 + epoch, variant.diversity_weight)
                val_loss, predictions, diagnostics = predict(model, valid, prepared, device, criterion)
                val_auc = base.auc(predictions)
                val_acc = float(accuracy_score([r["label"] for r in predictions],
                                               [int(r["probability"] >= 0.5) for r in predictions]))
                epoch_row = {"fold": fold, "epoch": epoch, "train_loss": train_loss,
                             "train_objective": train_objective,
                             "train_attention_overlap": train_overlap,
                             "valid_loss": val_loss, "valid_auc": val_auc,
                             "valid_accuracy_0.5": val_acc, "lr": optimizer.param_groups[0]["lr"],
                             "elapsed_seconds": round(time.monotonic() - epoch_start, 2)}
                history.append(epoch_row)
                if not all(math.isfinite(float(value)) for value in (
                    train_loss, train_objective, val_loss, val_auc, val_acc)):
                    raise ValueError(f"Nonfinite history: {variant.name} seed={seed} fold={fold} epoch={epoch}")
                improved = (val_auc > best_auc + 1e-12 or
                            (abs(val_auc - best_auc) <= 1e-12 and val_loss < best_loss))
                if improved:
                    best_auc, best_loss, best_epoch, stale = val_auc, val_loss, epoch, 0
                    torch.save(model.state_dict(), seed_root / f"fold_{fold}.pt")
                    base.write_csv(seed_root / f"fold_{fold}_oof.csv",
                                   [{"fold": fold, **r} for r in predictions],
                                   ["fold", "slide_id", "label", "probability"])
                    base.write_json(seed_root / f"fold_{fold}_anchor_diagnostics.json",
                                    {"fold": fold, "epoch": epoch, "slides": diagnostics})
                else:
                    stale += 1
                scheduler.step()
                print(f"{variant.name} seed={seed} fold={fold} epoch={epoch} "
                      f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
                      f"val_auc={val_auc:.4f} best={best_auc:.4f}", flush=True)
                if stale >= PATIENCE:
                    break
            base.write_csv(seed_root / f"fold_{fold}_history.csv", history, list(history[0]))
            selected = base.read_csv(seed_root / f"fold_{fold}_oof.csv")
            selected = [{"fold": int(r["fold"]), "slide_id": r["slide_id"],
                         "label": int(r["label"]), "probability": float(r["probability"])}
                        for r in selected]
            seed_oof.extend(selected)
            all_oof.extend({"seed": seed, **r} for r in selected)
            model.load_state_dict(torch.load(seed_root / f"fold_{fold}.pt",
                                             map_location=device, weights_only=True))
            train_eval_loss, train_predictions, _ = predict(
                model, train, prepared, device, criterion, diagnostics_enabled=False)
            train_acc = float(accuracy_score([r["label"] for r in train_predictions],
                                             [int(r["probability"] >= 0.5) for r in train_predictions]))
            labels = [r["label"] for r in selected]
            classes = [int(r["probability"] >= 0.5) for r in selected]
            tn, fp, fn, tp = confusion_matrix(labels, classes, labels=[0, 1]).ravel()
            fold_summary = {"fold": fold, "best_epoch": best_epoch,
                            "validation_auc": best_auc, "validation_loss": best_loss,
                            "training_loss_at_best_checkpoint": train_eval_loss,
                            "training_accuracy_0.5": train_acc,
                            "validation_accuracy_0.5": float(accuracy_score(labels, classes)),
                            "validation_balanced_accuracy_0.5": float(balanced_accuracy_score(labels, classes)),
                            "validation_sensitivity_0.5": float(tp / (tp + fn)),
                            "validation_specificity_0.5": float(tn / (tn + fp)),
                            "elapsed_seconds": round(time.monotonic() - fold_start, 1)}
            fold_summaries.append(fold_summary)
            runs.append({"seed": seed, **fold_summary})
            base.write_json(seed_root / f"fold_{fold}_summary.json", fold_summary)
            if len(runs) == 1:
                print(f"First fold took {fold_summary['elapsed_seconds']:.1f}s; "
                      f"90-fold linear projection {fold_summary['elapsed_seconds'] * 90 / 3600:.2f}h "
                      "(early stopping and variants may differ).", flush=True)
            del model, optimizer, scheduler
            if device.type == "cuda":
                torch.cuda.empty_cache()
        seed_oof.sort(key=lambda r: r["slide_id"])
        if len(seed_oof) != 270 or len({r["slide_id"] for r in seed_oof}) != 270:
            raise ValueError(f"Incomplete OOF output: {variant.name}, seed {seed}")
        base.write_csv(seed_root / "oof_predictions.csv", seed_oof,
                       ["fold", "slide_id", "label", "probability"])
        mean_auc = float(np.mean([r["validation_auc"] for r in fold_summaries]))
        seed_means.append(mean_auc)
        base.write_json(seed_root / "cv_summary.json",
                        {"folds": fold_summaries, "mean_validation_auc": mean_auc,
                         "std_validation_auc": float(np.std([r["validation_auc"] for r in fold_summaries], ddof=1)),
                         "oof_auc": base.auc(seed_oof), "test_evaluated": False})
    base.write_csv(out / "oof_predictions_all_seeds.csv", all_oof,
                   ["seed", "fold", "slide_id", "label", "probability"])
    ensemble = ensemble_oof(all_oof)
    base.write_csv(out / "oof_predictions_ensemble.csv", ensemble,
                   ["slide_id", "fold", "label", "probability"])
    summary = {"variant": variant.__dict__, "seeds": list(SEEDS), "runs": runs,
               "seed_mean_validation_auc": seed_means,
               "mean_validation_auc": float(np.mean(seed_means)),
               "std_seed_mean_validation_auc": float(np.std(seed_means, ddof=1)),
               "oof_ensemble_auc": base.auc(ensemble), "test_evaluated": False}
    base.write_json(out / "cv_summary.json", summary)
    verify_new_variant(out, variant, rows)
    return summary


def ensemble_oof(all_oof: list[dict]):
    grouped = {}
    for row in all_oof:
        grouped.setdefault(row["slide_id"], []).append(row)
    if len(grouped) != 270 or any(len(values) != 3 or
                                  {int(x["seed"]) for x in values} != set(SEEDS) or
                                  len({int(x["fold"]) for x in values}) != 1 or
                                  len({int(x["label"]) for x in values}) != 1
                                  for values in grouped.values()):
        raise ValueError("OOF must contain three seeds on the same fold for each of 270 slides")
    return [{"slide_id": sid, "fold": int(values[0]["fold"]),
             "label": int(values[0]["label"]),
             "probability": float(np.mean([float(x["probability"]) for x in values]))}
            for sid, values in sorted(grouped.items())]


def verify_new_variant(out: Path, variant: Variant, rows: list[dict]):
    run = json.loads((out / "cv_summary.json").read_text())
    oof = base.read_csv(out / "oof_predictions_all_seeds.csv")
    expected = {r["slide_id"]: (r["label"], r["validation_fold"])
                for r in rows if r["official_split"] == "train"}
    if len(run["runs"]) != 15 or len(oof) != 810:
        raise ValueError(f"Incomplete variant: {variant.name}")
    observed = set()
    for r in oof:
        key = (int(r["seed"]), r["slide_id"])
        if key in observed or key[0] not in SEEDS or r["slide_id"] not in expected:
            raise ValueError(f"Invalid OOF key: {key}")
        observed.add(key)
        if (int(r["label"]), int(r["fold"])) != expected[r["slide_id"]]:
            raise ValueError(f"OOF label/fold mismatch: {key}")
        p = float(r["probability"])
        if not math.isfinite(p) or not 0.0 <= p <= 1.0:
            raise ValueError(f"Invalid OOF probability: {key}")
    for seed in SEEDS:
        for fold in FOLDS:
            directory = out / f"seed_{seed}"
            for suffix in (".pt", "_history.csv", "_summary.json", "_oof.csv", "_anchor_diagnostics.json"):
                if not (directory / f"fold_{fold}{suffix}").is_file():
                    raise FileNotFoundError(f"Missing fold artifact: {variant.name} {seed} {fold} {suffix}")
            diag = json.loads((directory / f"fold_{fold}_anchor_diagnostics.json").read_text())
            if len(diag["slides"]) != 54:
                raise ValueError(f"Missing diagnostics: {variant.name} {seed} {fold}")
    if run["test_evaluated"] is not False:
        raise ValueError("Test evaluation found in validation output")


def read_reference(bundle: Path, expected_sha: str, rows: list[dict], prepared_hash: str):
    if sha256(bundle) != expected_sha:
        raise ValueError("Reference bundle SHA-256 mismatch")
    expected_slides = {r["slide_id"]: (r["label"], r["validation_fold"])
                       for r in rows if r["official_split"] == "train"}
    result = {}
    with zipfile.ZipFile(bundle) as archive:
        manifest = json.loads(archive.read("reference_manifest.json"))
        if manifest["kind"] != "ab_phase2_phase3_validation_reference" or manifest["member_count"] != 72:
            raise ValueError("Invalid reference bundle manifest")
        if set(archive.namelist()) != set(manifest["member_sha256"]) | {"reference_manifest.json"}:
            raise ValueError("Unexpected or missing reference member")
        for name, digest in manifest["member_sha256"].items():
            if hashlib.sha256(archive.read(name)).hexdigest() != digest:
                raise ValueError(f"Reference member checksum mismatch: {name}")
        for phase, variant in (("phase2", PHASE2), ("phase3", PHASE3)):
            phase_info = manifest["phases"][phase]
            if (phase_info["code_sha256"] != BASELINE_SHA256 or
                    phase_info["prepared_manifest_sha256"] != prepared_hash or
                    phase_info["anchors"] != 4 or
                    phase_info["diversity_weight"] != variant.diversity_weight):
                raise ValueError(f"Reference provenance mismatch: {phase}")
            config = json.loads(archive.read(f"{phase}/config.json"))
            if (config["seeds"] != list(SEEDS) or config["folds"] != 5 or
                    config["anchors"] != 4 or config["diversity_weight"] != variant.diversity_weight or
                    config["manifest_sha256"] != prepared_hash):
                raise ValueError(f"Reference root settings mismatch: {phase}")
            summary = json.loads(archive.read(f"{phase}/cv_summary.json"))
            if len(summary["runs"]) != 15 or summary["test_evaluated"] is not False:
                raise ValueError(f"Incomplete reference summary: {phase}")
            text = archive.read(f"{phase}/oof_predictions_all_seeds.csv").decode()
            oof = list(csv.DictReader(io.StringIO(text)))
            validate_oof(oof, expected_slides)
            diagnostics = []
            for seed in SEEDS:
                seed_config = json.loads(archive.read(f"{phase}/seed_{seed}/config.json"))
                if any(seed_config[key] != value for key, value in {
                    "seed": seed, "anchors": 4, "diversity_weight": variant.diversity_weight,
                    "epochs": EPOCHS, "patience": PATIENCE, "lr": LR,
                    "weight_decay": WEIGHT_DECAY, "manifest_sha256": prepared_hash,
                }.items()):
                    raise ValueError(f"Reference seed settings mismatch: {phase} {seed}")
                for fold in FOLDS:
                    f = json.loads(archive.read(f"{phase}/seed_{seed}/fold_{fold}_summary.json"))
                    matching = [r for r in summary["runs"] if r["seed"] == seed and r["fold"] == fold]
                    if len(matching) != 1 or matching[0]["validation_auc"] != f["validation_auc"]:
                        raise ValueError(f"Reference fold summary mismatch: {phase} {seed} {fold}")
                    diagnostic = json.loads(archive.read(
                        f"{phase}/seed_{seed}/fold_{fold}_anchor_diagnostics.json"))
                    if diagnostic["fold"] != fold or len(diagnostic["slides"]) != 54:
                        raise ValueError(f"Reference fold diagnostics missing: {phase} {seed} {fold}")
                    for slide in diagnostic["slides"]:
                        sid = slide["slide_id"]
                        if sid not in expected_slides or expected_slides[sid][1] != fold:
                            raise ValueError(f"Reference diagnostic slide mismatch: {sid}")
                        diagnostics.append({"seed": seed, "fold": fold,
                                            "label": expected_slides[sid][0], **slide})
            result[variant.name] = {"variant": variant, "summary": summary,
                                    "oof": oof, "diagnostics": diagnostics, "source": "archive"}
    if any(len(v["diagnostics"]) != 810 for v in result.values()):
        raise ValueError("Incomplete archived diagnostics")
    return result


def validate_oof(oof: list[dict], expected_slides: dict):
    if len(oof) != 810:
        raise ValueError("Expected 810 OOF rows")
    seen = set()
    for row in oof:
        seed, sid = int(row["seed"]), row["slide_id"]
        if seed not in SEEDS or sid not in expected_slides or (seed, sid) in seen:
            raise ValueError(f"Invalid OOF identity: {seed} {sid}")
        seen.add((seed, sid))
        if (int(row["label"]), int(row["fold"])) != expected_slides[sid]:
            raise ValueError(f"OOF label or fold mismatch: {seed} {sid}")
        p = float(row["probability"])
        if not math.isfinite(p) or not 0.0 <= p <= 1.0:
            raise ValueError(f"Invalid OOF probability: {seed} {sid}")


def load_new_result(root: Path, variant: Variant, rows: list[dict]):
    out = root / variant.name
    verify_new_variant(out, variant, rows)
    expected = {r["slide_id"]: (r["label"], r["validation_fold"])
                for r in rows if r["official_split"] == "train"}
    summary = json.loads((out / "cv_summary.json").read_text())
    oof = base.read_csv(out / "oof_predictions_all_seeds.csv")
    validate_oof(oof, expected)
    diagnostics = []
    for seed in SEEDS:
        for fold in FOLDS:
            data = json.loads((out / f"seed_{seed}" / f"fold_{fold}_anchor_diagnostics.json").read_text())
            for slide in data["slides"]:
                diagnostics.append({"seed": seed, "fold": fold, **slide})
    return {"variant": variant, "summary": summary, "oof": oof,
            "diagnostics": diagnostics, "source": "new"}


def preflight(prepared: Path, rows: list[dict], root: Path, device: torch.device):
    base.seed_all(42)
    reference = base.MILNet(anchors=4).to(device)
    new = ABMILNet(PHASE2).to(device)
    new.load_state_dict(reference.state_dict(), strict=True)
    x = torch.randn(9, 512, device=device)
    for model in (reference, new):
        model.zero_grad(set_to_none=True)
    old_instance, old_bag, old_attention, old_vec = reference(x)
    new_instance, new_bag, new_attention, new_vec, _ = new(x)
    for left, right in ((old_instance, new_instance), (old_bag, new_bag),
                        (old_attention, new_attention), (old_vec, new_vec)):
        torch.testing.assert_close(left, right, rtol=1e-6, atol=1e-7)
    old_loss = 0.5 * F.binary_cross_entropy_with_logits(old_bag, torch.ones_like(old_bag)) + 0.5 * F.binary_cross_entropy_with_logits(old_instance.max().view(1, 1), torch.ones_like(old_bag))
    new_loss = 0.5 * F.binary_cross_entropy_with_logits(new_bag, torch.ones_like(new_bag)) + 0.5 * F.binary_cross_entropy_with_logits(new_instance.max().view(1, 1), torch.ones_like(new_bag))
    old_loss.backward()
    new_loss.backward()
    for (name_old, old_param), (name_new, new_param) in zip(reference.named_parameters(), new.named_parameters()):
        assert name_old == name_new
        torch.testing.assert_close(old_param.grad, new_param.grad, rtol=1e-6, atol=1e-7)
    for alpha in (0.0, 0.25, 0.5, 0.75):
        for n in (1, 3, 7, 33):
            feats = torch.randn(n, 512, device=device)
            feats[0].zero_()
            scores = torch.ones(n, device=device) if n == 7 else torch.arange(n, device=device).float()
            original_ids = torch.arange(n - 1, -1, -1, device=device)
            selected, _ = select_diverse(feats, scores, original_ids, alpha, 4)
            assert len(selected) == min(4, n) and len(set(selected.cpu().tolist())) == len(selected)
            if alpha == 0.0:
                torch.testing.assert_close(selected, torch.topk(scores, k=min(4, n), sorted=True).indices)
    fusion = ABMILNet(NEW_VARIANTS[3]).to(device)
    fusion.load_state_dict(reference.state_dict(), strict=False)
    _, fused_logit, _, _, details = fusion(x)
    torch.testing.assert_close(fused_logit, old_bag.detach(), rtol=1e-6, atol=1e-6)
    torch.testing.assert_close(details["fusion_weights"], torch.full((4,), 0.25, device=device), rtol=0, atol=0)
    criterion = nn.BCEWithLogitsLoss()
    train = [r for r in rows if r["official_split"] == "train"]
    sampled = [min(train, key=lambda row: row["n_patches"]),
               max(train, key=lambda row: row["n_patches"])]
    observations = []
    for row in sampled:
        feats = base.load_bag(prepared, row, device)
        for variant in (NEW_VARIANTS[2], NEW_VARIANTS[3], NEW_VARIANTS[5]):
            base.seed_all(42)
            model, optimizer, scheduler = make_model(variant, device)
            optimizer.zero_grad(set_to_none=True)
            loss, _, _, attention, _ = bag_step(model, feats, row["label"], criterion)
            objective = loss + variant.diversity_weight * base.attention_overlap(attention)
            if not torch.isfinite(objective):
                raise ValueError(f"Nonfinite preflight objective: {variant.name} {row['slide_id']}")
            objective.backward()
            if any(not torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None):
                raise ValueError(f"Nonfinite preflight gradient: {variant.name} {row['slide_id']}")
            checkpoint = root / "preflight_checkpoint.pt"
            torch.save(model.state_dict(), checkpoint)
            model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
            checkpoint.unlink()
            observations.append({"variant": variant.name, "slide_id": row["slide_id"],
                                 "n_patches": len(feats), "objective": float(objective.item())})
            del model, optimizer, scheduler
            if device.type == "cuda":
                torch.cuda.empty_cache()
    base.write_json(root / "preflight.json", {"checks": observations,
                    "alpha0_exact_phase2_forward_and_gradients": True,
                    "fusion_zero_initially_uniform": True, "test_bags_loaded": False})


def mean_field(rows: list[dict], field: str, label=None):
    values = [float(r[field]) for r in rows if r.get(field) is not None and
              (label is None or int(r["label"]) == label)]
    return float(np.mean(values)) if values else None


def analyze(root: Path, results: dict):
    expected_keys = {PHASE2.name, PHASE3.name} | {v.name for v in NEW_VARIANTS}
    if set(results) != expected_keys:
        raise ValueError("Analysis requires six new configurations and two archives")
    identity = None
    comparison, all_ensembles = [], {}
    for name, data in results.items():
        runs = data["summary"]["runs"]
        if len(runs) != 15 or len(data["oof"]) != 810 or len(data["diagnostics"]) != 810:
            raise ValueError(f"Incomplete comparison input: {name}")
        if any(not math.isfinite(float(r["validation_auc"])) for r in runs):
            raise ValueError(f"Nonfinite fold AUC in {name}")
        keys = {(int(r["seed"]), r["slide_id"], int(r["label"]), int(r["fold"]))
                for r in data["oof"]}
        if identity is None:
            identity = keys
        elif keys != identity:
            raise ValueError(f"OOF cohorts differ: {name}")
        ensemble = ensemble_oof(data["oof"])
        all_ensembles[name] = ensemble
        labels = [r["label"] for r in ensemble]
        predictions = [int(r["probability"] >= 0.5) for r in ensemble]
        tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
        weights = [r.get("fusion_weights") for r in data["diagnostics"]]
        weight_departure = [float(np.mean(np.abs(np.asarray(w) - 0.25)))
                            for w in weights if w is not None and len(w) == 4]
        comparison.append({
            "variant": name, "source": data["source"], "anchors": data["variant"].anchors,
            "alpha": data["variant"].alpha, "fusion": data["variant"].fusion,
            "diversity_weight": data["variant"].diversity_weight,
            "fold_seed_runs": len(runs),
            "mean_validation_auc": float(np.mean([r["validation_auc"] for r in runs])),
            "sd_validation_auc_descriptive": float(np.std([r["validation_auc"] for r in runs], ddof=1)),
            "mean_training_loss_at_best": mean_field(runs, "training_loss_at_best_checkpoint"),
            "mean_validation_loss_at_best": mean_field(runs, "validation_loss"),
            "mean_training_accuracy_0.5_at_best": mean_field(runs, "training_accuracy_0.5"),
            "mean_validation_accuracy_0.5_at_best": mean_field(runs, "validation_accuracy_0.5"),
            "oof_ensemble_auc": base.auc(ensemble),
            "oof_ensemble_accuracy_0.5": float(accuracy_score(labels, predictions)),
            "oof_ensemble_balanced_accuracy_0.5": float(balanced_accuracy_score(labels, predictions)),
            "oof_ensemble_sensitivity_0.5": float(tp / (tp + fn)),
            "oof_ensemble_specificity_0.5": float(tn / (tn + fp)),
            "positive_oof_mean_probability": mean_field(ensemble, "probability", 1),
            "negative_oof_mean_probability": mean_field(ensemble, "probability", 0),
            "mean_selected_feature_cosine": mean_field(data["diagnostics"], "mean_pairwise_anchor_cosine"),
            "mean_attention_cosine": mean_field(data["diagnostics"], "mean_pairwise_attention_cosine"),
            "mean_anchor_bag_cosine": mean_field(data["diagnostics"], "mean_pairwise_anchor_bag_cosine"),
            "positive_attention_cosine": mean_field(data["diagnostics"], "mean_pairwise_attention_cosine", 1),
            "negative_attention_cosine": mean_field(data["diagnostics"], "mean_pairwise_attention_cosine", 0),
            "mean_fusion_weight_entropy": mean_field(data["diagnostics"], "fusion_weight_entropy"),
            "mean_absolute_fusion_weight_departure_from_quarter": float(np.mean(weight_departure)) if weight_departure else None,
        })
    order = [PHASE2.name, *[v.name for v in NEW_VARIANTS], PHASE3.name]
    comparison.sort(key=lambda row: order.index(row["variant"]))
    base.write_csv(root / "comparison_table.csv", comparison, list(comparison[0]))
    slide_rows = [{"variant": name, "source": results[name]["source"],
                   "slide_id": row["slide_id"], "fold": row["fold"],
                   "label": row["label"], "probability": row["probability"],
                   "prediction_0.5": int(row["probability"] >= 0.5)}
                  for name in order for row in all_ensembles[name]]
    base.write_csv(root / "oof_predictions_all_variants.csv", slide_rows, list(slide_rows[0]))

    paired_contrasts = [
        ("A1_alpha_025_vs_Phase2", "A1_alpha_025", PHASE2.name),
        ("A1_alpha_050_vs_Phase2", "A1_alpha_050", PHASE2.name),
        ("A1_alpha_075_vs_Phase2", "A1_alpha_075", PHASE2.name),
        ("A2_vs_Phase2", "A2_learned_fusion", PHASE2.name),
        ("B_penalty_K2", "B_K2_L01", "B_K2_L0"),
        ("B_penalty_K4", PHASE3.name, PHASE2.name),
        ("B_K2_vs_K4_L0", "B_K2_L0", PHASE2.name),
        ("B_K2_vs_K4_L01", "B_K2_L01", PHASE3.name),
    ]
    run_map = {name: {(int(r["seed"]), int(r["fold"])): r for r in data["summary"]["runs"]}
               for name, data in results.items()}
    paired = []
    for contrast, left, right in paired_contrasts:
        for seed in SEEDS:
            for fold in FOLDS:
                delta = (float(run_map[left][seed, fold]["validation_auc"]) -
                         float(run_map[right][seed, fold]["validation_auc"]))
                paired.append({"contrast": contrast, "left": left, "right": right,
                               "seed": seed, "fold": fold, "auc_delta": delta})
    base.write_csv(root / "paired_fold_auc_differences.csv", paired, list(paired[0]))
    interactions = []
    for seed in SEEDS:
        for fold in FOLDS:
            k2 = (run_map["B_K2_L01"][seed, fold]["validation_auc"] -
                  run_map["B_K2_L0"][seed, fold]["validation_auc"])
            k4 = (run_map[PHASE3.name][seed, fold]["validation_auc"] -
                  run_map[PHASE2.name][seed, fold]["validation_auc"])
            interactions.append({"seed": seed, "fold": fold,
                                 "penalty_effect_K2": k2, "penalty_effect_K4": k4,
                                 "K_by_penalty_interaction": k2 - k4})
    base.write_csv(root / "B_interaction.csv", interactions, list(interactions[0]))
    alpha_candidates = [row for row in comparison if row["variant"] in
                        {PHASE2.name, "A1_alpha_025", "A1_alpha_050", "A1_alpha_075"}]
    selected = max(alpha_candidates, key=lambda row: (row["mean_validation_auc"], -row["alpha"]))
    alpha_report = {"selection_metric": "mean of 15 fold/seed validation AUC values",
                    "tie_rule": "smaller alpha", "selected_alpha": selected["alpha"],
                    "selected_variant": selected["variant"],
                    "candidates": [{"alpha": row["alpha"],
                                    "mean_validation_auc": row["mean_validation_auc"],
                                    "oof_ensemble_auc": row["oof_ensemble_auc"]}
                                   for row in alpha_candidates],
                    "test_evaluated": False}
    base.write_json(root / "A1_alpha_selection.json", alpha_report)
    report = {"scope": "validation only", "new_fold_seed_runs": 90,
              "reference_fold_seed_runs": 30, "test_evaluated": False,
              "alpha_selection": alpha_report,
              "A2_paired_mean_auc_delta_vs_Phase2": mean_field(
                  [r for r in paired if r["contrast"] == "A2_vs_Phase2"], "auc_delta"),
              "B_mean_interaction": mean_field(interactions, "K_by_penalty_interaction"),
              "interpretation": "Paired validation comparisons are descriptive; no official-test result is produced."}
    base.write_json(root / "AB_validation_summary.json", report)
    for name, ensemble in all_ensembles.items():
        if results[name]["source"] == "new":
            continue
        archive_out = root / "reference" / name
        archive_out.mkdir(parents=True, exist_ok=True)
        base.write_csv(archive_out / "oof_predictions_ensemble.csv", ensemble,
                       ["slide_id", "fold", "label", "probability"])
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--prepared-root", type=Path, required=True)
    parser.add_argument("--output-root", type=Path, required=True)
    parser.add_argument("--reference-bundle", type=Path, required=True)
    parser.add_argument("--reference-sha256", required=True)
    args = parser.parse_args()
    runner_hash = sha256(Path(__file__))
    if sha256(Path(base.__file__)) != BASELINE_SHA256:
        raise ValueError("The imported DSMIL baseline differs from the archived Phase 2/3 source")
    prepared_hash = sha256(args.prepared_root / "slides.csv")
    if prepared_hash != PREPARED_SHA256:
        raise ValueError(f"Prepared manifest differs from archived Phase 2/3: {prepared_hash}")
    rows, info = base.load_manifest(args.prepared_root)
    refs = read_reference(args.reference_bundle, args.reference_sha256, rows, prepared_hash)
    if args.output_root.exists():
        raise FileExistsError(f"Choose a fresh output directory: {args.output_root}")
    root = args.output_root
    root.mkdir(parents=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = {"kind": "pre_phase4_AB_validation", "prepared_manifest_sha256": prepared_hash,
              "baseline_sha256": BASELINE_SHA256, "runner_sha256": runner_hash,
              "reference_sha256": args.reference_sha256, "seeds": list(SEEDS),
              "folds": list(FOLDS), "new_variants": [v.__dict__ for v in NEW_VARIANTS],
              "reference_variants": [PHASE2.__dict__, PHASE3.__dict__],
              "device": str(device), "test_evaluated": False}
    base.write_json(root / "run_config.json", config)
    shutil.copy2(args.reference_bundle, root / "reference_bundle.zip")
    preflight(args.prepared_root, rows, root, device)
    for variant in NEW_VARIANTS:
        print(f"Starting {variant.name}: 3 seeds x 5 folds", flush=True)
        train_variant(variant, rows, info, args.prepared_root, root, device,
                      prepared_hash, runner_hash)
    results = {**refs}
    results.update({v.name: load_new_result(root, v, rows) for v in NEW_VARIANTS})
    report = analyze(root, results)
    checkpoints = list(root.glob("A*/seed_*/fold_*.pt")) + list(root.glob("B_K2*/seed_*/fold_*.pt"))
    histories = list(root.glob("A*/seed_*/fold_*_history.csv")) + list(root.glob("B_K2*/seed_*/fold_*_history.csv"))
    if len(checkpoints) != 90 or len(histories) != 90:
        raise ValueError("Missing checkpoints or histories in final output")
    if list(root.rglob("test_predictions*")) or list(root.rglob("test_summary.json")):
        raise ValueError("Test evaluation artifacts are forbidden in this run")
    base.write_json(root / "COMPLETE.json", {"status": "complete", "new_runs": 90,
                    "new_oof_rows": 4860, "reference_runs": 30,
                    "reference_sha256": args.reference_sha256,
                    "prepared_manifest_sha256": prepared_hash,
                    "test_evaluated": False})
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase4_a1_runner.py
"""Stage-specific A1 test reporting and Phase-4 UNI validation.

Separate notebooks invoke the A1 test stage and the validation-only UNI stages.
The UNI models retain A1's four-anchor rank selector and objective.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import time
import zipfile
from pathlib import Path

import numpy as np
import sklearn
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from torch import nn
from torch.nn import functional as F

import baseline_dsmil as base
import ab_validation_runner as ab


SEEDS = (42, 43, 44)
FOLDS = range(5)
UNISCALE_MANIFEST_SHA256 = ab.PREPARED_SHA256
EPOCHS, PATIENCE = 50, 10
LR, WEIGHT_DECAY = 1e-4, 1e-3
ANCHORS, CANDIDATES, TOP_C = 4, 32, 32
NEARBY_SCALE_PATCHES = 8.0
GAMMA_GRID = (0.0, 0.25, 0.5, 1.0)


def digest(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def checked_reference(bundle: Path, destination: Path) -> Path:
    """Extract only the small, checksummed A1 test reference."""
    with zipfile.ZipFile(bundle) as archive:
        manifest = json.loads(archive.read("reference_manifest.json"))
        members = manifest["member_sha256"]
        if (manifest.get("kind") != "a1_alpha025_test_reference" or
                set(archive.namelist()) != set(members) | {"reference_manifest.json"}):
            raise ValueError("Invalid A1 reference bundle")
        if len([x for x in members if x.endswith(".pt")]) != 15:
            raise ValueError("A1 reference requires 15 checkpoints")
        destination.mkdir(parents=True, exist_ok=False)
        for name, expected in members.items():
            relative = Path(name)
            if relative.is_absolute() or ".." in relative.parts:
                raise ValueError(f"Unsafe bundle member: {name}")
            payload = archive.read(name)
            if hashlib.sha256(payload).hexdigest() != expected:
                raise ValueError(f"A1 bundle member checksum mismatch: {name}")
            target = destination / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(payload)
    return destination


def checked_reference_directory(source: Path, manifest_path: Path,
                                destination: Path) -> Path:
    """Copy and verify the archived A1 files from an attached Kaggle input."""
    if not source.is_dir():
        raise ValueError(f"Missing A1 reference directory: {source}")
    manifest = json.loads(manifest_path.read_text())
    members = manifest.get("member_sha256", {})
    if (manifest.get("kind") != "a1_alpha025_test_reference" or
            len([name for name in members if name.endswith(".pt")]) != 15):
        raise ValueError("Invalid A1 reference manifest")
    destination.mkdir(parents=True, exist_ok=False)
    for name, expected in members.items():
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"Unsafe A1 reference member: {name}")
        path = source / relative
        if not path.is_file():
            raise ValueError(f"Missing A1 reference member: {path}")
        payload = path.read_bytes()
        if hashlib.sha256(payload).hexdigest() != expected:
            raise ValueError(f"A1 reference checksum mismatch: {path}")
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
    return destination


def load_uni_manifest(prepared: Path) -> tuple[list[dict], dict]:
    info = json.loads((prepared / "PREPROCESSING_COMPLETE.json").read_text())
    if any(info.get(k) != v for k, v in {
            "source": "uni", "feature_dim": 1024, "seed": 42, "folds": 5,
            "slides": 399, "train_slides": 270, "test_slides": 129}.items()):
        raise ValueError("UNI prepared completion metadata does not match the study protocol")
    rows = base.read_csv(prepared / "slides.csv")
    if len(rows) != 399 or len({r["slide_id"] for r in rows}) != 399:
        raise ValueError("Expected 399 distinct UNI slides")
    for row in rows:
        sid = row["slide_id"]
        row["label"] = int(row["label"])
        row["n_patches"] = int(row["n_patches"])
        if row["label"] not in (0, 1):
            raise ValueError(f"Invalid UNI label: {sid}")
        if row["official_split"] == "train":
            row["validation_fold"] = int(row["validation_fold"])
            if not 0 <= row["validation_fold"] < 5 or sid.startswith("test_"):
                raise ValueError(f"Invalid UNI training fold: {sid}")
        elif row["official_split"] == "test":
            if row["validation_fold"] or not sid.startswith("test_"):
                raise ValueError(f"Invalid UNI official-test row: {sid}")
            row["validation_fold"] = ""
        else:
            raise ValueError(f"Invalid UNI split: {sid}")
        if row["n_patches"] < 1 or int(row["feature_dim"]) != 1024:
            raise ValueError(f"Invalid UNI patch count/dimension: {sid}")
        if row["official_split"] == "test":
            # Phase-4 runs are validation-only: do not open UNI test feature files.
            continue
        for field in ("bag_path", "coords_path"):
            rel = Path(row[field])
            if not row[field] or rel.is_absolute() or ".." in rel.parts or not (prepared / rel).is_file():
                raise ValueError(f"Missing or unsafe UNI {field}: {sid}")
        # Check .npy headers without loading all 4.9M patch rows into memory.
        bag = np.load(prepared / row["bag_path"], mmap_mode="r", allow_pickle=False)
        coords = np.load(prepared / row["coords_path"], mmap_mode="r", allow_pickle=False)
        if (bag.shape != (row["n_patches"], 1024) or bag.dtype != np.float32 or
                coords.shape != (row["n_patches"], 2) or coords.dtype != np.int64):
            raise ValueError(f"Invalid UNI bag/coordinate shape or dtype: {sid}")
        del bag, coords
    train = [r for r in rows if r["official_split"] == "train"]
    test = [r for r in rows if r["official_split"] == "test"]
    if (len(train) != 270 or len(test) != 129 or
            any(sum(r["validation_fold"] == fold for r in train) != 54 for fold in FOLDS) or
            next(r["label"] for r in rows if r["slide_id"] == "test_114") != 0):
        raise ValueError("UNI cohort, folds, or test_114 label mismatch")
    return rows, info


def load_coordinate_metadata(path: Path, rows: list[dict]) -> dict[str, dict]:
    audit = json.loads(path.read_text())
    by_id = {r["slide_id"]: r for r in audit["slides"]}
    train = [r for r in rows if r["official_split"] == "train"]
    if len(by_id) != 270 or set(by_id) != {r["slide_id"] for r in train}:
        raise ValueError("UNI training coordinate audit slide coverage mismatch")
    for row in train:
        item = by_id[row["slide_id"]]
        dim = item["level_dim"]
        if (len(dim) != 2 or min(dim) <= 0 or item.get("patch_size_px", 0) <= 0 or
                item.get("patch_level") != 0 or item["patches"] != row["n_patches"] or
                item["coordinate_shape"] != [row["n_patches"], 2] or
                item["out_of_bounds_coordinate_rows"] != 0):
            raise ValueError(f"Invalid source coordinate metadata: {row['slide_id']}")
    return by_id


def load_uni(prepared: Path, row: dict, device: torch.device,
             metadata: dict[str, dict], spatial: bool) -> tuple[torch.Tensor, torch.Tensor | None]:
    array = np.load(prepared / row["bag_path"], mmap_mode="r", allow_pickle=False)
    if array.shape != (row["n_patches"], 1024) or array.dtype != np.float32:
        raise ValueError(f"Invalid UNI feature array: {row['slide_id']}")
    features = torch.tensor(array, dtype=torch.float32, device=device)
    if not torch.isfinite(features).all():
        raise ValueError(f"Nonfinite UNI features: {row['slide_id']}")
    if not spatial:
        return features, None
    raw = np.load(prepared / row["coords_path"], mmap_mode="r", allow_pickle=False)
    dim = np.asarray(metadata[row["slide_id"]]["level_dim"], dtype=np.float32)
    if (raw.shape != (len(features), 2) or raw.dtype != np.int64 or
            np.any(raw < 0) or np.any(raw >= dim)):
        raise ValueError(f"Invalid UNI coordinates or bounds: {row['slide_id']}")
    coords = torch.tensor(np.asarray(raw, dtype=np.float32) /
                          float(metadata[row["slide_id"]]["patch_size_px"]),
                          dtype=torch.float32, device=device)
    return features, coords


def select_spatial_anchors(features: torch.Tensor, scores: torch.Tensor,
                           original_ids: torch.Tensor, coords: torch.Tensor):
    """A1 top-32 candidate pool with 0.50 score/0.25 feature/0.25 space ranks."""
    if len(features) < 1 or coords.shape != (len(features), 2):
        raise ValueError("Spatial selection needs paired feature and coordinate rows")
    k = min(ANCHORS, len(features))
    values = scores.detach().cpu().numpy()
    ids = original_ids.detach().cpu().numpy()
    candidates = np.lexsort((ids, -values))[:min(CANDIDATES, len(features))].tolist()
    chosen = [candidates[0]]
    records = [{"score_rank": 1, "feature_rank": None, "spatial_rank": None}]
    normalized = F.normalize(features.detach()[candidates], p=2, dim=1, eps=1e-12)
    cosine = (normalized @ normalized.T).cpu().numpy()
    candidate_coords = coords.detach()[candidates]
    distance = torch.linalg.vector_norm(
        candidate_coords[:, None, :] - candidate_coords[None, :, :], dim=2).cpu().numpy()
    position = {row: local for local, row in enumerate(candidates)}
    while len(chosen) < k:
        remaining = [row for row in candidates if row not in chosen]
        score_rank = {row: rank for rank, row in enumerate(remaining, 1)}
        feature_gap = {row: 1.0 - max(float(cosine[position[row], position[j]])
                                      for j in chosen) for row in remaining}
        spatial_gap = {row: min(float(distance[position[row], position[j]])
                                for j in chosen) for row in remaining}
        feature_order = sorted(remaining, key=lambda row: (-feature_gap[row], int(ids[row])))
        spatial_order = sorted(remaining, key=lambda row: (-spatial_gap[row], int(ids[row])))
        feature_rank = {row: rank for rank, row in enumerate(feature_order, 1)}
        spatial_rank = {row: rank for rank, row in enumerate(spatial_order, 1)}
        winner = min(remaining, key=lambda row: (
            0.50 * score_rank[row] + 0.25 * feature_rank[row] + 0.25 * spatial_rank[row],
            -float(values[row]), int(ids[row])))
        chosen.append(winner)
        records.append({"score_rank": score_rank[winner],
                        "feature_rank": feature_rank[winner],
                        "spatial_rank": spatial_rank[winner]})
    return torch.as_tensor(chosen, dtype=torch.long, device=features.device), records


def top_c_mask(biased_logits: torch.Tensor, original_ids: torch.Tensor,
               limit: int) -> torch.Tensor:
    """Choose the highest combined appearance-and-distance scores per anchor."""
    n, k = biased_logits.shape
    keep = min(limit, n)
    ids = original_ids.detach().cpu().numpy()
    scores = biased_logits.detach().cpu().numpy()
    mask = torch.zeros((n, k), dtype=torch.bool, device=biased_logits.device)
    for column in range(k):
        ordering = np.lexsort((ids, -scores[:, column]))[:keep]
        mask[torch.as_tensor(ordering, device=biased_logits.device), column] = True
    return mask


class A1BagClassifier(base.BClassifier):
    def __init__(self, dimension: int, mode: str = "none", gamma: float = 0.0):
        super().__init__(dimension, 1, anchors=ANCHORS)
        if mode not in ("none", "relative_all", "relative_top32") or gamma < 0:
            raise ValueError("Invalid Phase-4 mode or gamma")
        self.mode, self.gamma = mode, gamma
        self.spatial = mode != "none"

    def forward(self, features, scores, original_ids=None, coords=None):
        if original_ids is None:
            original_ids = torch.arange(len(features), device=features.device)
        if self.spatial and (coords is None or coords.shape != (len(features), 2)):
            raise ValueError("Spatial A1 requires one x/y coordinate per feature")
        q = self.q(features).view(len(features), -1)
        if self.spatial:
            indices, ranks = select_spatial_anchors(features, scores[:, 0], original_ids, coords)
        else:
            indices, ranks = ab.select_diverse(features, scores[:, 0], original_ids, 0.25, ANCHORS)
        anchor_queries = q.index_select(0, indices)
        content_logits = (q @ anchor_queries.T) / math.sqrt(128)
        if self.spatial:
            distance = torch.linalg.vector_norm(
                coords[:, None, :] - coords.index_select(0, indices)[None, :, :], dim=2)
            spatial_penalty = self.gamma * torch.clamp(
                (distance / NEARBY_SCALE_PATCHES).square(), max=1.0)
            logits = content_logits - spatial_penalty
        else:
            logits = content_logits
        if self.mode == "relative_top32":
            support = top_c_mask(logits, original_ids, TOP_C)
            logits = logits.masked_fill(~support, -torch.inf)
        else:
            support = torch.ones_like(logits, dtype=torch.bool)
        attention = F.softmax(logits, dim=0)
        anchor_bags = attention.T @ self.v(features)
        bag = anchor_bags.mean(dim=0, keepdim=True).view(1, 1, features.shape[1])
        prediction = self.fcc(bag).view(1, -1)
        return prediction, attention, {"indices": indices, "rank_records": ranks,
                                       "anchor_bags": anchor_bags,
                                       "support_count": support.sum(dim=0),
                                       "support_contains_anchor": support[indices, torch.arange(len(indices), device=indices.device)]}


class A1UniNet(nn.Module):
    def __init__(self, mode: str = "none", gamma: float = 0.0):
        super().__init__()
        self.i_classifier = base.FCLayer(1024, 1)
        self.b_classifier = A1BagClassifier(1024, mode, gamma)

    def forward(self, features, original_ids=None, coords=None):
        features, scores = self.i_classifier(features)
        bag_logit, attention, details = self.b_classifier(features, scores, original_ids, coords)
        return scores, bag_logit, attention, details


def make_uni_model(mode: str, gamma: float, device: torch.device):
    # Restore RNG after construction so identical seeds initialize shared A1
    # weights identically in all three variants.
    rng_cpu = torch.get_rng_state()
    rng_cuda = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    model = A1UniNet(mode, gamma).to(device)
    torch.set_rng_state(rng_cpu)
    if rng_cuda is not None:
        torch.cuda.set_rng_state_all(rng_cuda)
    for layer in model.modules():
        if isinstance(layer, (nn.Linear, nn.Conv1d)):
            nn.init.orthogonal_(layer.weight)
            if layer.bias is not None:
                nn.init.constant_(layer.bias, 0)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.5, 0.9),
                                 weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS, eta_min=5e-6)
    return model, optimizer, scheduler


def uni_step(model, features, label, criterion, original_ids=None, coords=None):
    scores, bag_logit, attention, details = model(features, original_ids, coords)
    target = torch.tensor([[float(label)]], device=features.device)
    loss = 0.5 * criterion(bag_logit, target) + 0.5 * criterion(scores.max().view(1, 1), target)
    return loss, float(torch.sigmoid(bag_logit.detach()).item()), scores, attention, details


def train_epoch(model, rows, prepared, device, metadata, optimizer, criterion, seed, epoch, fold):
    model.train()
    order = np.random.default_rng(seed + fold * 100000 + epoch).permutation(len(rows))
    total = 0.0
    for index in order:
        row = rows[int(index)]
        features, coords = load_uni(prepared, row, device, metadata, model.b_classifier.spatial)
        permutation = torch.randperm(len(features), device=device)
        features = features[permutation]
        if coords is not None:
            coords = coords[permutation]
        optimizer.zero_grad(set_to_none=True)
        loss, _, _, _, _ = uni_step(model, features, row["label"], criterion, permutation, coords)
        if not torch.isfinite(loss):
            raise ValueError(f"Nonfinite objective: {row['slide_id']}")
        loss.backward()
        optimizer.step()
        total += float(loss.detach().cpu())
    return total / len(rows)


@torch.no_grad()
def predict_uni(model, rows, prepared, device, metadata, criterion, diagnostics=False):
    model.eval()
    predictions, details_rows = [], []
    total = 0.0
    for row in rows:
        features, coords = load_uni(prepared, row, device, metadata,
                                    model.b_classifier.spatial or diagnostics)
        loss, probability, logits, attention, details = uni_step(
            model, features, row["label"], criterion, coords=coords)
        total += float(loss.detach().cpu())
        predictions.append({"slide_id": row["slide_id"], "label": row["label"],
                            "probability": probability})
        if diagnostics:
            idx = details["indices"]
            k = len(idx)
            selected = F.normalize(features[idx], p=2, dim=1)
            bags = F.normalize(details["anchor_bags"], p=2, dim=1)
            off = ~torch.eye(k, dtype=torch.bool, device=device)
            patch_px = int(metadata[row["slide_id"]]["patch_size_px"])
            pairwise = []
            for left in range(k):
                for right in range(left + 1, k):
                    cosine = float(torch.dot(selected[left], selected[right]).item())
                    distance_patches = float(torch.linalg.vector_norm(
                        coords[idx[left]] - coords[idx[right]]).item())
                    pairwise.append({"anchor_left": left, "anchor_right": right,
                                     "feature_cosine_dissimilarity": 1.0 - cosine,
                                     "spatial_distance_patch_widths": distance_patches,
                                     "spatial_distance_pixels": distance_patches * patch_px})
            anchor_distances = torch.linalg.vector_norm(
                coords[:, None, :] - coords.index_select(0, idx)[None, :, :], dim=2)
            weighted_radius = (attention * anchor_distances).sum(dim=0)
            details_rows.append({
                "slide_id": row["slide_id"], "label": row["label"],
                "anchor_row_indices": idx.cpu().tolist(),
                "anchor_instance_logits": logits[idx, 0].cpu().tolist(),
                "anchor_rank_records": details["rank_records"],
                "anchor_coordinates_pixels": torch.round(coords[idx] * patch_px).to(torch.int64).cpu().tolist(),
                "anchor_pairwise": pairwise,
                "min_anchor_spatial_distance_patch_widths": min(
                    (p["spatial_distance_patch_widths"] for p in pairwise), default=None),
                "mean_anchor_feature_cosine_dissimilarity": float(np.mean(
                    [p["feature_cosine_dissimilarity"] for p in pairwise])) if pairwise else None,
                "attention_support_counts": details["support_count"].cpu().tolist(),
                "attention_support_contains_anchor": details["support_contains_anchor"].cpu().tolist(),
                "attention_column_sums": attention.sum(dim=0).cpu().tolist(),
                "attention_weighted_radius_patch_widths": weighted_radius.cpu().tolist(),
                "mean_pairwise_anchor_cosine": float((selected @ selected.T)[off].mean()) if k > 1 else None,
                "mean_pairwise_anchor_bag_cosine": float((bags @ bags.T)[off].mean()) if k > 1 else None,
                "mean_pairwise_attention_cosine": float(base.attention_overlap(attention)),
            })
    return total / len(rows), predictions, details_rows


def test_metrics(predictions: list[dict], threshold: float, origin: str) -> dict:
    labels = [int(r["label"]) for r in predictions]
    classes = [int(float(r["probability"]) >= threshold) for r in predictions]
    tn, fp, fn, tp = confusion_matrix(labels, classes, labels=[0, 1]).ravel()
    return {"origin": origin, "test_slides": len(predictions), "n_models": 15,
            "threshold_from_oof": threshold, "test_auc": base.auc(predictions),
            "test_accuracy": float(accuracy_score(labels, classes)),
            "test_balanced_accuracy": float(balanced_accuracy_score(labels, classes)),
            "test_sensitivity": float(tp / (tp + fn)),
            "test_specificity": float(tn / (tn + fp)),
            "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
            "test_114_label": 0,
            "ensemble_rule": "mean of 15 checkpoint bag probabilities"}


def write_test(out: Path, per_checkpoint: list[dict], test: list[dict], threshold: float,
               origin: str) -> dict:
    if len(per_checkpoint) != 129 * 15:
        raise ValueError(f"Expected 1935 test predictions, got {len(per_checkpoint)}")
    expected = {r["slide_id"]: r["label"] for r in test}
    by_slide = {sid: [] for sid in expected}
    for item in per_checkpoint:
        sid = item["slide_id"]
        if sid not in expected or item["label"] != expected[sid]:
            raise ValueError(f"Invalid test slide/label: {sid}")
        if not math.isfinite(item["probability"]) or not 0 <= item["probability"] <= 1:
            raise ValueError(f"Invalid test probability: {sid}")
        by_slide[sid].append(item)
    if any(len(values) != 15 or {(r["seed"], r["fold"]) for r in values} !=
           {(s, f) for s in SEEDS for f in FOLDS} for values in by_slide.values()):
        raise ValueError("Incomplete test ensemble coverage")
    ensemble = [{"slide_id": sid, "label": expected[sid],
                 "probability": float(np.mean([r["probability"] for r in by_slide[sid]]))}
                for sid in sorted(expected)]
    for row in ensemble:
        row["prediction"] = int(row["probability"] >= threshold)
    base.write_csv(out / "test_predictions_per_checkpoint.csv", per_checkpoint,
                   ["seed", "fold", "slide_id", "label", "probability"])
    base.write_csv(out / "test_predictions_ensemble.csv", ensemble,
                   ["slide_id", "label", "probability", "prediction"])
    summary = test_metrics(ensemble, threshold, origin)
    base.write_json(out / "test_summary.json", summary)
    return summary


def evaluate_a1_reference(reference: Path, prepared: Path, rows: list[dict],
                          output: Path, device: torch.device) -> dict:
    config = json.loads((reference / "config.json").read_text())
    summary = json.loads((reference / "cv_summary.json").read_text())
    variant = config["variant"]
    if (variant != ab.Variant("A1_alpha_025", 4, alpha=0.25).__dict__ or
            config["prepared_manifest_sha256"] != UNISCALE_MANIFEST_SHA256 or
            config["runner_sha256"] != digest(Path(ab.__file__)) or
            config["baseline_sha256"] != digest(Path(base.__file__)) or
            config["seeds"] != list(SEEDS) or config["folds"] != list(FOLDS) or
            config["test_evaluated"] is not False or len(summary["runs"]) != 15 or
            summary["variant"] != variant or summary["seeds"] != list(SEEDS)):
        raise ValueError("A1 archived training contract mismatch")
    oof = base.read_csv(reference / "oof_predictions_ensemble.csv")
    train = {r["slide_id"]: r for r in rows if r["official_split"] == "train"}
    if len(oof) != 270 or len({r["slide_id"] for r in oof}) != 270 or {r["slide_id"] for r in oof} != set(train):
        raise ValueError("Incomplete A1 OOF ensemble")
    for row in oof:
        original = train[row["slide_id"]]
        if (int(row["label"]), int(row["fold"])) != (original["label"], original["validation_fold"]):
            raise ValueError(f"A1 OOF identity mismatch: {row['slide_id']}")
        row["label"] = int(row["label"])
        row["probability"] = float(row["probability"])
        if not math.isfinite(row["probability"]) or not 0 <= row["probability"] <= 1:
            raise ValueError(f"Invalid A1 OOF probability: {row['slide_id']}")
    if not math.isclose(base.auc(oof), summary["oof_ensemble_auc"], abs_tol=1e-12):
        raise ValueError("A1 OOF AUC mismatch")
    threshold = base.threshold_from_oof(oof)
    test = [r for r in rows if r["official_split"] == "test"]
    output.mkdir(parents=True, exist_ok=False)
    criterion = nn.BCEWithLogitsLoss()
    all_predictions = []
    for seed in SEEDS:
        for fold in FOLDS:
            model = ab.ABMILNet(ab.Variant("A1_alpha_025", 4, alpha=0.25)).to(device)
            model.load_state_dict(torch.load(reference / f"seed_{seed}" / f"fold_{fold}.pt",
                                             map_location=device, weights_only=True), strict=True)
            _, predicted, _ = ab.predict(model, test, prepared, device, criterion,
                                         diagnostics_enabled=False)
            all_predictions.extend({"seed": seed, "fold": fold, **r} for r in predicted)
            del model
            if device.type == "cuda":
                torch.cuda.empty_cache()
            print(f"A1 existing checkpoint test: seed={seed} fold={fold}", flush=True)
    result = write_test(output, all_predictions, test, threshold, "A1 alpha=0.25; archived 512-d uniscale")
    base.write_json(output / "COMPLETE.json", {"status": "complete", "test_summary": result,
                                                "archived_cv_source": str(reference),
                                                "prepared_manifest_sha256": digest(prepared / "slides.csv")})
    return result


def train_uni_variant(name: str, mode: str, gamma: float, rows: list[dict], prepared: Path,
                      metadata: dict[str, dict], output: Path, device: torch.device) -> dict:
    output.mkdir(parents=True, exist_ok=False)
    config = {"name": name, "feature_source": "uni", "feature_dim": 1024,
              "a1_alpha": 0.25, "anchors": 4, "candidate_pool": 32,
              "attention_diversity_weight": 0.0, "mode": mode, "gamma": gamma,
              "anchor_rank_weights": ([0.5, 0.25, 0.25] if mode != "none" else [0.75, 0.25]),
              "relative_distance_unit": "source patch widths" if mode != "none" else None,
              "nearby_scale_patch_widths": NEARBY_SCALE_PATCHES if mode != "none" else None,
              "attention_top_c": TOP_C if mode == "relative_top32" else None,
              "top_c_rule": "highest appearance-minus-distance logits, then masked softmax" if mode == "relative_top32" else None,
              "seeds": list(SEEDS), "folds": list(FOLDS), "epochs": EPOCHS,
              "patience": PATIENCE, "lr": LR, "weight_decay": WEIGHT_DECAY,
              "betas": [0.5, 0.9], "eta_min": 5e-6,
              "checkpoint": "max validation AUC; tie by min validation loss",
              "prepared_manifest_sha256": digest(prepared / "slides.csv"),
              "runner_sha256": digest(Path(__file__)),
              "versions": {"torch": torch.__version__, "numpy": np.__version__,
                           "sklearn": sklearn.__version__},
              "coordinate_pairing": "Assumed row-aligned from UNI source .pt/.h5; not independently proven"}
    base.write_json(output / "config.json", config)
    criterion = nn.BCEWithLogitsLoss()
    all_oof, fold_results = [], []
    for seed in SEEDS:
        seed_dir = output / f"seed_{seed}"
        seed_dir.mkdir()
        for fold in FOLDS:
            train = [r for r in rows if r["official_split"] == "train" and r["validation_fold"] != fold]
            valid = [r for r in rows if r["official_split"] == "train" and r["validation_fold"] == fold]
            if len(train) != 216 or len(valid) != 54:
                raise ValueError(f"Invalid UNI fold size: {fold}")
            base.seed_all(seed)
            model, optimizer, scheduler = make_uni_model(mode, gamma, device)
            best_auc, best_loss, best_epoch, stale = -1.0, float("inf"), 0, 0
            history = []
            start = time.monotonic()
            for epoch in range(1, EPOCHS + 1):
                train_loss = train_epoch(model, train, prepared, device, metadata,
                                         optimizer, criterion, seed, epoch, fold)
                valid_loss, predicted, _ = predict_uni(model, valid, prepared, device,
                                                       metadata, criterion)
                valid_auc = base.auc(predicted)
                valid_acc = float(accuracy_score([r["label"] for r in predicted],
                                                 [int(r["probability"] >= 0.5) for r in predicted]))
                if not all(map(math.isfinite, (train_loss, valid_loss, valid_auc, valid_acc))):
                    raise ValueError(f"Nonfinite training metric: {name} seed={seed} fold={fold}")
                history.append({"seed": seed, "fold": fold, "epoch": epoch,
                                "train_loss": train_loss, "valid_loss": valid_loss,
                                "valid_auc": valid_auc, "valid_accuracy_0.5": valid_acc,
                                "lr": optimizer.param_groups[0]["lr"]})
                improved = (valid_auc > best_auc + 1e-12 or
                            (abs(valid_auc - best_auc) <= 1e-12 and valid_loss < best_loss))
                if improved:
                    best_auc, best_loss, best_epoch, stale = valid_auc, valid_loss, epoch, 0
                    torch.save(model.state_dict(), seed_dir / f"fold_{fold}.pt")
                    base.write_csv(seed_dir / f"fold_{fold}_oof.csv",
                                   [{"seed": seed, "fold": fold, **r} for r in predicted],
                                   ["seed", "fold", "slide_id", "label", "probability"])
                else:
                    stale += 1
                scheduler.step()
                print(f"{name} seed={seed} fold={fold} epoch={epoch} "
                      f"train={train_loss:.4f} val={valid_loss:.4f} auc={valid_auc:.4f}", flush=True)
                if stale >= PATIENCE:
                    break
            base.write_csv(seed_dir / f"fold_{fold}_history.csv", history, list(history[0]))
            model.load_state_dict(torch.load(seed_dir / f"fold_{fold}.pt",
                                             map_location=device, weights_only=True))
            train_eval_loss, train_pred, train_diagnostics = predict_uni(
                model, train, prepared, device, metadata, criterion, diagnostics=True)
            _, _, validation_diagnostics = predict_uni(
                model, valid, prepared, device, metadata, criterion, diagnostics=True)
            if (len(train_diagnostics) != 216 or len(validation_diagnostics) != 54 or
                    any(len(item["anchor_pairwise"]) != 6 for item in
                        train_diagnostics + validation_diagnostics)):
                raise ValueError(f"Incomplete anchor-pair diagnostics: {name} {seed} {fold}")
            base.write_json(seed_dir / f"fold_{fold}_anchor_diagnostics.json",
                            {"epoch": best_epoch, "train_slides": train_diagnostics,
                             "validation_slides": validation_diagnostics})
            selected = base.read_csv(seed_dir / f"fold_{fold}_oof.csv")
            selected = [{"seed": int(r["seed"]), "fold": int(r["fold"]),
                         "slide_id": r["slide_id"], "label": int(r["label"]),
                         "probability": float(r["probability"])} for r in selected]
            all_oof.extend(selected)
            fold_result = {"seed": seed, "fold": fold, "best_epoch": best_epoch,
                           "validation_auc": best_auc, "validation_loss": best_loss,
                           "training_loss_at_best_checkpoint": train_eval_loss,
                           "training_auc_at_best_checkpoint": base.auc(train_pred),
                           "training_accuracy_0.5": float(accuracy_score(
                               [r["label"] for r in train_pred],
                               [int(r["probability"] >= 0.5) for r in train_pred])),
                           "validation_accuracy_0.5": float(accuracy_score(
                               [r["label"] for r in selected],
                               [int(r["probability"] >= 0.5) for r in selected])),
                           "elapsed_seconds": round(time.monotonic() - start, 1)}
            fold_results.append(fold_result)
            base.write_json(seed_dir / f"fold_{fold}_summary.json", fold_result)
            del model, optimizer, scheduler
            if device.type == "cuda":
                torch.cuda.empty_cache()
    all_oof.sort(key=lambda r: (r["seed"], r["slide_id"]))
    base.write_csv(output / "oof_predictions_all_seeds.csv", all_oof,
                   ["seed", "fold", "slide_id", "label", "probability"])
    ensemble = ab.ensemble_oof(all_oof)
    base.write_csv(output / "oof_predictions_ensemble.csv", ensemble,
                   ["slide_id", "fold", "label", "probability"])
    cv = {"name": name, "mode": mode, "gamma": gamma, "runs": fold_results,
          "mean_validation_auc": float(np.mean([r["validation_auc"] for r in fold_results])),
          "std_validation_auc": float(np.std([r["validation_auc"] for r in fold_results], ddof=1)),
          "mean_validation_loss": float(np.mean([r["validation_loss"] for r in fold_results])),
          "oof_ensemble_auc": base.auc(ensemble),
          "test_evaluated": False}
    base.write_json(output / "cv_summary.json", cv)
    base.write_json(output / "COMPLETE.json", {"status": "complete", "cv": cv,
                                                "checkpoints": 15, "oof_rows": 810,
                                                "anchor_diagnostic_slide_evaluations": 4050,
                                                "test_evaluated": False})
    return cv


def run_gamma_family(name: str, mode: str, rows: list[dict], prepared: Path,
                     metadata: dict[str, dict], output: Path, device: torch.device) -> dict:
    output.mkdir(parents=True, exist_ok=False)
    candidates = []
    for gamma in GAMMA_GRID:
        tag = f"gamma_{int(round(gamma * 100)):03d}"
        cv = train_uni_variant(f"{name}_{tag}", mode, gamma, rows, prepared,
                               metadata, output / tag, device)
        candidates.append({"gamma": gamma, "output": tag,
                           "mean_validation_auc": cv["mean_validation_auc"],
                           "mean_validation_loss": cv["mean_validation_loss"],
                           "oof_ensemble_auc": cv["oof_ensemble_auc"]})
    # Use validation AUC, then validation loss, then the smaller gamma.
    best = max(candidates, key=lambda row: (
        row["mean_validation_auc"], -row["mean_validation_loss"], -row["gamma"]))
    result = {"name": name, "mode": mode, "gamma_grid": list(GAMMA_GRID),
              "selection_rule": "highest mean validation AUC across 15 fits; tie by lower mean validation loss, then smaller gamma",
              "selected_gamma": best["gamma"], "selected_output": best["output"],
              "candidates": candidates, "test_evaluated": False}
    base.write_json(output / "gamma_tuning_summary.json", result)
    base.write_csv(output / "gamma_tuning_table.csv", candidates, list(candidates[0]))
    return result


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--stage", choices=("a1-test", "uni", "spatial-a", "spatial-b"), required=True)
    parser.add_argument("--uniscale-prepared", type=Path)
    parser.add_argument("--uni-prepared", type=Path)
    parser.add_argument("--a1-reference-bundle", type=Path)
    parser.add_argument("--a1-reference-dir", type=Path)
    parser.add_argument("--a1-reference-manifest", type=Path)
    parser.add_argument("--uni-audit", type=Path)
    parser.add_argument("--output-root", type=Path, required=True)
    args = parser.parse_args()
    if args.stage in ("a1-test", "uni"):
        if args.output_root.exists():
            raise FileExistsError(f"Choose a new output root: {args.output_root}")
        args.output_root.mkdir(parents=True)
    elif not (args.output_root / "Phase4_1_UNI_A1/COMPLETE.json").is_file():
        raise ValueError("UNI feature validation must complete before spatial validation")
    if (args.stage == "spatial-b" and not
            (args.output_root / "Phase4_2a_relative_all/gamma_tuning_summary.json").is_file()):
        raise ValueError("Experiment 2(a) must complete before 2(b)")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if args.stage == "a1-test":
        if (args.uniscale_prepared is None or
                (args.a1_reference_bundle is None) == (args.a1_reference_dir is None)):
            raise ValueError("A1 test requires prepared uniscale data and one reference source")
        uniscale_rows, _ = base.load_manifest(args.uniscale_prepared)
        if digest(args.uniscale_prepared / "slides.csv") != UNISCALE_MANIFEST_SHA256:
            raise ValueError("Uniscale manifest differs from A1 training data")
        if args.a1_reference_dir is not None:
            if args.a1_reference_manifest is None:
                raise ValueError("An A1 reference directory requires its checksum manifest")
            reference = checked_reference_directory(
                args.a1_reference_dir, args.a1_reference_manifest,
                args.output_root / "a1_reference")
        else:
            reference = checked_reference(args.a1_reference_bundle,
                                          args.output_root / "a1_reference")
        print("Archived A1 alpha=0.25 official test evaluation", flush=True)
        a1_test = evaluate_a1_reference(reference, args.uniscale_prepared, uniscale_rows,
                                        args.output_root / "A1_uniscale_test", device)
        base.write_json(args.output_root / "COMPLETE.json", {
            "status": "complete", "a1_test": a1_test, "archived_checkpoints_tested": 15})
        print("A1 TEST RESULT", json.dumps(a1_test, indent=2), flush=True)
        return
    if args.uni_prepared is None or args.uni_audit is None:
        raise ValueError("Phase-4 validation requires prepared UNI data and coordinate metadata")
    uni_rows, _ = load_uni_manifest(args.uni_prepared)
    metadata = load_coordinate_metadata(args.uni_audit, uni_rows)
    if args.stage == "uni":
        print("Phase 4 experiment 1: UNI features with A1 selector", flush=True)
        result = train_uni_variant("Phase4_1_UNI_A1", "none", 0.0, uni_rows, args.uni_prepared,
                                   metadata, args.output_root / "Phase4_1_UNI_A1", device)
        print("UNI VALIDATION RESULT", json.dumps(result, indent=2), flush=True)
        return
    if args.stage == "spatial-a":
        print("Phase 4 experiment 2(a): relative-distance attention over all patches", flush=True)
        result = run_gamma_family("Phase4_2a_relative_all", "relative_all", uni_rows,
                                  args.uni_prepared, metadata,
                                  args.output_root / "Phase4_2a_relative_all", device)
        print("EXPERIMENT 2(a) GAMMA SELECTION", json.dumps(result, indent=2), flush=True)
        return
    print("Phase 4 experiment 2(b): relative-distance attention over top-32 biased scores", flush=True)
    spatial_b = run_gamma_family("Phase4_2b_relative_top32", "relative_top32", uni_rows,
                                 args.uni_prepared, metadata,
                                 args.output_root / "Phase4_2b_relative_top32", device)
    uni_cv = json.loads((args.output_root / "Phase4_1_UNI_A1/cv_summary.json").read_text())
    spatial_a = json.loads((args.output_root / "Phase4_2a_relative_all/gamma_tuning_summary.json").read_text())
    base.write_json(args.output_root / "COMPLETE.json", {
        "status": "complete", "sequence": ["Phase4_1_UNI_A1", "Phase4_2a_relative_all",
                                          "Phase4_2b_relative_top32"],
        "uni_cv": uni_cv, "spatial_a_tuning": spatial_a, "spatial_b_tuning": spatial_b,
        "total_new_fits": 15 * (1 + 2 * len(GAMMA_GRID)), "test_evaluated": False})
    print("ALL STEPS COMPLETE", args.output_root, flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/multiscale_a1_runner.py
"""Fixed and gated multiscale A1 validation; frozen three-variant test evaluation."""
from __future__ import annotations

import argparse
import hashlib
import json
import math
import shutil
import time
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix
from torch import nn
from torch.nn import functional as F

import baseline_dsmil as base
import ab_validation_runner as ab
import phase4_a1_runner as p4

SEEDS = (42, 43, 44)
FOLDS = range(5)
EPOCHS, PATIENCE = 50, 10
VARIANTS = ("M-fixed", "M-adaptive")


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_multi(prepared):
    info = json.loads((prepared / "PREPROCESSING_COMPLETE.json").read_text())
    required = {"source": "multiscale", "feature_dim": 1024, "seed": 42,
                "folds": 5, "slides": 399, "train_slides": 270, "test_slides": 129,
                "multiscale_columns": {"20x": [0, 512], "5x": [512, 1024]}}
    if any(info.get(k) != v for k, v in required.items()):
        raise ValueError("Multiscale completion metadata mismatch")
    rows = base.read_csv(prepared / "slides.csv")
    if len(rows) != 399 or len({r["slide_id"] for r in rows}) != 399:
        raise ValueError("Expected 399 unique multiscale slides")
    for r in rows:
        r["label"] = int(r["label"])
        r["n_patches"] = int(r["n_patches"])
        if r["label"] not in (0, 1) or int(r["feature_dim"]) != 1024 or r["n_patches"] < 1:
            raise ValueError(f"Invalid multiscale row: {r['slide_id']}")
        rel = Path(r["bag_path"])
        if rel.is_absolute() or ".." in rel.parts or not (prepared / rel).is_file():
            raise ValueError(f"Missing or unsafe bag: {r['slide_id']}")
        if r["official_split"] == "train":
            r["validation_fold"] = int(r["validation_fold"])
            if not 0 <= r["validation_fold"] < 5 or r["slide_id"].startswith("test_"):
                raise ValueError("Invalid train fold")
        elif r["official_split"] == "test":
            if r["validation_fold"] or not r["slide_id"].startswith("test_"):
                raise ValueError("Invalid test fold")
        else:
            raise ValueError("Invalid split")
    if [sum(r["official_split"] == "train" and r["validation_fold"] == f for r in rows)
            for f in FOLDS] != [54] * 5:
        raise ValueError("Fold sizes mismatch")
    if sum(r["official_split"] == "test" for r in rows) != 129:
        raise ValueError("Test size mismatch")
    if next(r["label"] for r in rows if r["slide_id"] == "test_114") != 0:
        raise ValueError("test_114 label mismatch")
    return rows, info


def check_cohort(uniscale_rows, multi_rows):
    def signature(rows):
        return {r["slide_id"]: (r["label"], r["official_split"],
                                r["validation_fold"] if r["official_split"] == "train" else None)
                for r in rows}
    if signature(uniscale_rows) != signature(multi_rows):
        raise ValueError("Uniscale/multiscale cohort or folds differ")


def bag(prepared, row, device):
    a = np.load(prepared / row["bag_path"], mmap_mode="r", allow_pickle=False)
    if a.shape != (row["n_patches"], 1024) or a.dtype != np.dtype("float32"):
        raise ValueError(f"Invalid bag shape: {row['slide_id']}")
    x = torch.tensor(a, dtype=torch.float32, device=device)
    if not torch.isfinite(x).all():
        raise ValueError(f"Nonfinite bag: {row['slide_id']}")
    return x


class MultiA1(nn.Module):
    def __init__(self, adaptive=False):
        super().__init__()
        variant = ab.Variant("A1_alpha_025", 4, alpha=0.25)
        self.i_classifier = base.FCLayer(1024, 1)
        self.b_classifier = ab.ABBClassifier(variant)
        # The original ABBClassifier has 512-dimensional layers; replace only its input-facing modules.
        self.b_classifier.q = nn.Sequential(nn.Linear(1024, 128), nn.ReLU(),
                                            nn.Linear(128, 128), nn.Tanh())
        self.b_classifier.fcc = nn.Conv1d(1, 1, kernel_size=1024)
        for layer in (self.i_classifier, self.b_classifier):
            for module in layer.modules():
                if isinstance(module, (nn.Linear, nn.Conv1d)):
                    nn.init.orthogonal_(module.weight)
                    nn.init.zeros_(module.bias)
        self.gate = nn.Linear(1024, 2) if adaptive else None
        if self.gate is not None:
            nn.init.zeros_(self.gate.weight)
            nn.init.zeros_(self.gate.bias)

    def forward(self, x):
        weights = None
        if self.gate is not None:
            weights = F.softmax(self.gate(x), dim=1)
            x = torch.cat((2 * weights[:, :1] * x[:, :512],
                           2 * weights[:, 1:] * x[:, 512:]), dim=1)
        feats, classes = self.i_classifier(x)
        prediction, attention, _, details = self.b_classifier(feats, classes)
        return classes, prediction, attention, details, weights


def make_model(adaptive, device):
    model = MultiA1(adaptive).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.5, 0.9), weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS, 5e-6)
    return model, optimizer, scheduler


def step(model, x, label, criterion):
    logits, bag_logit, attention, details, gate = model(x)
    target = torch.tensor([[float(label)]], device=x.device)
    loss = .5 * criterion(bag_logit.view(1, 1), target) + .5 * criterion(logits.max().view(1, 1), target)
    return loss, float(torch.sigmoid(bag_logit).item()), details, gate


@torch.no_grad()
def predict(model, rows, prepared, device, criterion, diagnostics=False):
    model.eval()
    records, extra, total = [], [], 0.0
    for r in rows:
        loss, prob, details, gate = step(model, bag(prepared, r, device), r["label"], criterion)
        records.append({"slide_id": r["slide_id"], "label": r["label"], "probability": prob})
        total += float(loss.item())
        if diagnostics:
            item = {"slide_id": r["slide_id"], "anchor_rows": details["indices"].cpu().tolist(),
                    "anchor_ranks": details["rank_records"]}
            if gate is not None:
                item["mean_gate_20x"] = float(gate[:, 0].mean().item())
                item["mean_gate_5x"] = float(gate[:, 1].mean().item())
                item["gate_20x_quantiles"] = torch.quantile(gate[:, 0],
                    torch.tensor([0., .25, .5, .75, 1.], device=device)).cpu().tolist()
                item["mean_gate_entropy"] = float((-(gate * gate.clamp_min(1e-12).log()).sum(1)).mean().item())
            extra.append(item)
    return total / len(rows), records, extra


def train(variant, prepared, out):
    if variant not in VARIANTS:
        raise ValueError(variant)
    rows, info = load_multi(prepared)
    out.mkdir(parents=True, exist_ok=False)
    config = {"variant": variant, "alpha": .25, "anchors": 4, "seeds": list(SEEDS),
              "folds": list(FOLDS), "epochs": EPOCHS, "patience": PATIENCE,
              "prepared_sha256": digest(prepared / "slides.csv"),
              "completion_sha256": digest(prepared / "PREPROCESSING_COMPLETE.json"),
              "runner_sha256": digest(__file__), "test_evaluated": False,
              "prepared_info": info}
    base.write_json(out / "config.json", config)
    criterion = nn.BCEWithLogitsLoss()
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    all_oof, summaries = [], []
    for seed in SEEDS:
        for fold in FOLDS:
            train_rows = [r for r in rows if r["official_split"] == "train" and r["validation_fold"] != fold]
            val_rows = [r for r in rows if r["official_split"] == "train" and r["validation_fold"] == fold]
            assert len(train_rows) == 216 and len(val_rows) == 54
            base.seed_all(seed)
            model, optimizer, scheduler = make_model(variant == "M-adaptive", device)
            run = out / f"seed_{seed}" / f"fold_{fold}"
            run.mkdir(parents=True)
            best_auc, best_loss, best_epoch, stale = -1., float("inf"), 0, 0
            history = []
            for epoch in range(1, EPOCHS + 1):
                model.train()
                order = np.random.default_rng(seed + fold * 100000 + epoch).permutation(len(train_rows))
                total = 0.
                for i in order:
                    row = train_rows[int(i)]
                    x = bag(prepared, row, device)
                    x = x[torch.randperm(len(x), device=device)]
                    optimizer.zero_grad(set_to_none=True)
                    loss, _, _, _ = step(model, x, row["label"], criterion)
                    if not torch.isfinite(loss):
                        raise ValueError("Nonfinite training loss")
                    loss.backward()
                    optimizer.step()
                    total += float(loss.item())
                val_loss, predictions, diagnostics = predict(model, val_rows, prepared, device, criterion, True)
                auc = base.auc(predictions)
                acc = accuracy_score([r["label"] for r in predictions],
                                     [int(r["probability"] >= .5) for r in predictions])
                history.append({"epoch": epoch, "train_loss": total / len(train_rows),
                                "val_loss": val_loss, "val_auc": auc, "val_accuracy_0.5": float(acc),
                                "lr": optimizer.param_groups[0]["lr"]})
                if not all(math.isfinite(float(v)) for v in history[-1].values()):
                    raise ValueError("Nonfinite history")
                if auc > best_auc + 1e-12 or abs(auc - best_auc) <= 1e-12 and val_loss < best_loss:
                    best_auc, best_loss, best_epoch, stale = auc, val_loss, epoch, 0
                    torch.save(model.state_dict(), run / "checkpoint.pt")
                    base.write_csv(run / "oof.csv", [{"seed": seed, "fold": fold, **r} for r in predictions],
                                   ["seed", "fold", "slide_id", "label", "probability"])
                    base.write_json(run / "diagnostics.json", {"slides": diagnostics})
                else:
                    stale += 1
                scheduler.step()
                print(f"{variant} seed={seed} fold={fold} epoch={epoch} val_auc={auc:.4f}", flush=True)
                if stale >= PATIENCE:
                    break
            base.write_csv(run / "history.csv", history, list(history[0]))
            model.load_state_dict(torch.load(run / "checkpoint.pt", map_location=device, weights_only=True))
            train_loss, train_preds, _ = predict(model, train_rows, prepared, device, criterion)
            selected = base.read_csv(run / "oof.csv")
            all_oof.extend(selected)
            summary = {"seed": seed, "fold": fold, "best_epoch": best_epoch,
                       "validation_auc": best_auc, "validation_loss": best_loss,
                       "training_loss_at_best": train_loss,
                       "training_accuracy_0.5": float(accuracy_score(
                           [r["label"] for r in train_preds],
                           [int(r["probability"] >= .5) for r in train_preds])),
                       "validation_accuracy_0.5": float(accuracy_score(
                           [int(r["label"]) for r in selected],
                           [int(float(r["probability"]) >= .5) for r in selected]))}
            base.write_json(run / "summary.json", summary)
            summaries.append(summary)
            del model
            if device.type == "cuda":
                torch.cuda.empty_cache()
    if len(all_oof) != 810 or len(summaries) != 15:
        raise ValueError("Incomplete CV")
    base.write_csv(out / "oof_all.csv", all_oof, ["seed", "fold", "slide_id", "label", "probability"])
    by_slide = {}
    for r in all_oof:
        by_slide.setdefault(r["slide_id"], []).append(r)
    if len(by_slide) != 270 or any(len(v) != 3 for v in by_slide.values()):
        raise ValueError("OOF slide coverage mismatch")
    ensemble = [{"slide_id": sid, "label": int(v[0]["label"]),
                 "probability": float(np.mean([float(r["probability"]) for r in v]))}
                for sid, v in sorted(by_slide.items())]
    base.write_csv(out / "oof_predictions_ensemble.csv", ensemble,
                   ["slide_id", "label", "probability"])
    base.write_csv(out / "fold_results.csv", summaries, list(summaries[0]))
    base.write_json(out / "COMPLETE.json", {"status": "complete", "variant": variant,
                                          "fold_seed_runs": 15, "oof_rows": 810,
                                          "oof_auc": base.auc(ensemble), "test_evaluated": False})


def verify_a1_results(prior, multi_rows):
    config = json.loads((prior / "a1_reference/config.json").read_text())
    summary = json.loads((prior / "A1_uniscale_test/test_summary.json").read_text())
    complete = json.loads((prior / "COMPLETE.json").read_text())
    if (config["variant"] != ab.Variant("A1_alpha_025", 4, alpha=.25).__dict__ or
            config["seeds"] != list(SEEDS) or config["folds"] != list(FOLDS) or
            complete["status"] != "complete" or complete["archived_checkpoints_tested"] != 15 or
            summary["test_slides"] != 129 or summary["n_models"] != 15 or
            summary["test_114_label"] != 0):
        raise ValueError("A1 prior test metadata mismatch")
    oof = base.read_csv(prior / "a1_reference/oof_predictions_ensemble.csv")
    if len(oof) != 270 or len({r["slide_id"] for r in oof}) != 270:
        raise ValueError("A1 prior OOF coverage mismatch")
    train_expected = {r["slide_id"]: (r["label"], r["validation_fold"])
                      for r in multi_rows if r["official_split"] == "train"}
    if (set(train_expected) != {r["slide_id"] for r in oof} or
            any((int(r["label"]), int(r["fold"])) != train_expected[r["slide_id"]]
                for r in oof)):
        raise ValueError("A1 OOF labels or folds differ from multiscale manifest")
    threshold = base.threshold_from_oof([{"label": int(r["label"]),
        "probability": float(r["probability"])} for r in oof])
    if not math.isclose(threshold, summary["threshold_from_oof"], abs_tol=1e-12):
        raise ValueError("A1 test threshold does not match OOF")
    expected = {r["slide_id"]: r["label"] for r in multi_rows if r["official_split"] == "test"}
    ensemble = base.read_csv(prior / "A1_uniscale_test/test_predictions_ensemble.csv")
    single = base.read_csv(prior / "A1_uniscale_test/test_predictions_per_checkpoint.csv")
    if (len(expected) != 129 or len(ensemble) != 129 or len(single) != 1935 or
            {r["slide_id"] for r in ensemble} != set(expected)):
        raise ValueError("A1 prior test coverage mismatch")
    groups = {sid: {} for sid in expected}
    for r in single:
        sid, key = r["slide_id"], (int(r["seed"]), int(r["fold"]))
        p = float(r["probability"])
        if sid not in expected or int(r["label"]) != expected[sid] or key in groups[sid] or not 0 <= p <= 1:
            raise ValueError("Invalid A1 checkpoint prediction")
        groups[sid][key] = p
    required = {(s, f) for s in SEEDS for f in FOLDS}
    for r in ensemble:
        sid, p = r["slide_id"], float(r["probability"])
        if (int(r["label"]) != expected[sid] or set(groups[sid]) != required or
                not math.isclose(p, float(np.mean(list(groups[sid].values()))), abs_tol=1e-12) or
                int(r["prediction"]) != int(p >= threshold)):
            raise ValueError("A1 ensemble is inconsistent with checkpoint predictions")
    records = [{"label": int(r["label"]), "probability": float(r["probability"])} for r in ensemble]
    if not math.isclose(base.auc(records), summary["test_auc"], abs_tol=1e-12):
        raise ValueError("A1 prior test AUC mismatch")
    return summary


def verify_variant(variant, prepared, output):
    config = json.loads((output / "config.json").read_text())
    complete = json.loads((output / "COMPLETE.json").read_text())
    if (config["variant"] != variant or config["alpha"] != .25 or
            config["prepared_sha256"] != digest(prepared / "slides.csv") or
            config["completion_sha256"] != digest(prepared / "PREPROCESSING_COMPLETE.json") or
            config["runner_sha256"] != digest(__file__) or
            complete["fold_seed_runs"] != 15 or complete["test_evaluated"] is not False):
        raise ValueError(f"Invalid saved variant: {variant}")
    for seed in SEEDS:
        for fold in FOLDS:
            if not (output / f"seed_{seed}" / f"fold_{fold}" / "checkpoint.pt").is_file():
                raise ValueError("Missing checkpoint")
    oof = base.read_csv(output / "oof_predictions_ensemble.csv")
    if len(oof) != 270 or len({r["slide_id"] for r in oof}) != 270:
        raise ValueError("Incomplete OOF")
    return [{"slide_id": r["slide_id"], "label": int(r["label"]),
             "probability": float(r["probability"])} for r in oof]


def final_test(multi, prior, fixed, adaptive, out):
    mrows, _ = load_multi(multi)
    verify_a1_results(prior, mrows)
    verify_variant("M-fixed", multi, fixed)
    verify_variant("M-adaptive", multi, adaptive)
    out.mkdir(parents=True, exist_ok=False)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    criterion = nn.BCEWithLogitsLoss()
    results = []
    shutil.copytree(prior / "A1_uniscale_test", out / "U-uniscale")
    old_summary = json.loads((out / "U-uniscale" / "test_summary.json").read_text())
    old_ensemble = base.read_csv(out / "U-uniscale" / "test_predictions_ensemble.csv")
    old_summary["test_pr_auc"] = float(average_precision_score(
        [int(r["label"]) for r in old_ensemble], [float(r["probability"]) for r in old_ensemble]))
    old_summary["test_accuracy_0.5"] = float(accuracy_score(
        [int(r["label"]) for r in old_ensemble],
        [int(float(r["probability"]) >= .5) for r in old_ensemble]))
    base.write_json(out / "U-uniscale" / "test_summary.json", old_summary)
    results.append({"variant": "U-uniscale", **old_summary})
    test = [r for r in mrows if r["official_split"] == "test"]
    for variant, source in (("M-fixed", fixed), ("M-adaptive", adaptive)):
        oof = verify_variant(variant, multi, source)
        threshold = base.threshold_from_oof(oof)
        all_preds = []
        for seed in SEEDS:
            for fold in FOLDS:
                model = MultiA1(variant == "M-adaptive").to(device)
                model.load_state_dict(torch.load(source / f"seed_{seed}" / f"fold_{fold}" / "checkpoint.pt",
                                                 map_location=device, weights_only=True), strict=True)
                _, preds, _ = predict(model, test, multi, device, criterion)
                all_preds.extend({"seed": seed, "fold": fold, **r} for r in preds)
                del model
                if device.type == "cuda":
                    torch.cuda.empty_cache()
        dest = out / variant
        dest.mkdir()
        summary = p4.write_test(dest, all_preds, test, threshold, variant)
        ensemble = base.read_csv(dest / "test_predictions_ensemble.csv")
        summary["test_pr_auc"] = float(average_precision_score(
            [int(r["label"]) for r in ensemble], [float(r["probability"]) for r in ensemble]))
        summary["test_accuracy_0.5"] = float(accuracy_score(
            [int(r["label"]) for r in ensemble], [int(float(r["probability"]) >= .5) for r in ensemble]))
        base.write_json(dest / "test_summary.json", summary)
        results.append({"variant": variant, **summary})
    base.write_csv(out / "test_comparison.csv", results, sorted(set().union(*(r.keys() for r in results))))
    base.write_json(out / "COMPLETE.json", {"status": "complete", "variants": [r["variant"] for r in results],
                                          "test_slides_each": 129, "models_per_variant": 15,
                                          "test_evaluated": True})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", choices=("validate_fixed", "validate_adaptive", "final_test"), required=True)
    ap.add_argument("--multi", type=Path, required=True)
    ap.add_argument("--prior-a1", type=Path)
    ap.add_argument("--fixed", type=Path)
    ap.add_argument("--adaptive", type=Path)
    ap.add_argument("--output", type=Path, required=True)
    args = ap.parse_args()
    if args.stage.startswith("validate"):
        train("M-fixed" if args.stage == "validate_fixed" else "M-adaptive", args.multi, args.output)
    else:
        final_test(args.multi, args.prior_a1, args.fixed, args.adaptive, args.output)


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, torch, zipfile

root = Path('/kaggle/working/multiscale-a1')
root.mkdir(parents=True, exist_ok=True)
manifest = {'A1_uniscale_test/COMPLETE.json': '2a6073f7c35e218ffed57f444964a88dd5354e0275e95df61190500b91e0bd09', 'A1_uniscale_test/test_predictions_ensemble.csv': '22680bdc83fbed6f3af99cb478f346c0be2e632ab368f3e3b13a74d846e80a32', 'A1_uniscale_test/test_predictions_per_checkpoint.csv': '61c213df7587ad84c8c86636f4cf38c007e04714e2bf3eaf017d522d3c142035', 'A1_uniscale_test/test_summary.json': 'fab9d069be9dcfbab970a99f2086ecfd04eceedc0950f0aba71cb60845fa02a6', 'COMPLETE.json': '097a5fb577a829bf88f8ec55822ef214c82de21e3eec8fef53cd428b08cd8be5', 'a1_reference/config.json': '7bd662e7522729bb6239a9816362021481a682e77222da723120f5c97acf8412', 'a1_reference/cv_summary.json': '7962988e0e31d4fc37452c8b972685b8c5f80b59231d0fc3a4264cafe75bbec1', 'a1_reference/oof_predictions_ensemble.csv': '47b61f88525d7d55ade1601610063ca4b756281193b972dcddad2d0f46edcea1'}
prior = root / 'prior-a1-test'
prior.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as archive:
    assert set(archive.namelist()) == set(manifest)
    for name, expected in manifest.items():
        rel = Path(name)
        assert not rel.is_absolute() and '..' not in rel.parts
        data = archive.read(name)
        assert hashlib.sha256(data).hexdigest() == expected, name
        target = prior / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)

sources = {}
for marker in Path('/kaggle/input').rglob('PREPROCESSING_COMPLETE.json'):
    info = json.loads(marker.read_text())
    if info.get('source') == 'multiscale':
        sources.setdefault(info['source'], []).append(marker.parent)
assert len(sources.get('multiscale', [])) == 1, sources
multiscale = sources['multiscale'][0]

import sys
sys.path.insert(0, '/kaggle/working')
import multiscale_a1_runner as runner
mrows, _ = runner.load_multi(multiscale)
runner.verify_a1_results(prior, mrows)
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle'
print('Inputs verified:', multiscale, 'and embedded prior A1 test results')
print('Visible CUDA devices:', torch.cuda.device_count())


In [ ]:
# Verify the adaptive extension starts from the exact fixed A1 model.
import torch
sample = torch.randn(11, 1024)
torch.manual_seed(123); fixed_probe = runner.MultiA1(False)
torch.manual_seed(123); adaptive_probe = runner.MultiA1(True)
fixed_result, adaptive_result = fixed_probe(sample), adaptive_probe(sample)
torch.testing.assert_close(fixed_result[1], adaptive_result[1], rtol=0, atol=0)
assert fixed_result[3]['indices'].tolist() == adaptive_result[3]['indices'].tolist()
torch.testing.assert_close(adaptive_result[4], torch.full((11, 2), 0.5))
loss, _, _, _ = runner.step(adaptive_probe, sample, 1, torch.nn.BCEWithLogitsLoss())
loss.backward()
assert adaptive_probe.gate.weight.grad is not None
assert adaptive_probe.gate.weight.grad.abs().sum().item() > 0
print('Fixed/adaptive initialization and gate gradient verified.')


In [ ]:
# Resource pilot on the largest official-training bag; no model selection.
import time
largest = max((r for r in mrows if r['official_split'] == 'train'), key=lambda r: r['n_patches'])
for adaptive, gpu in ((False, 0), (True, 1 if torch.cuda.device_count() > 1 else 0)):
    device = torch.device(f'cuda:{gpu}')
    # Cumulative peaks give a conservative bound on reruns and avoid resetting
    # an allocator that may not yet be initialized for this device.
    started = time.monotonic()
    probe = runner.MultiA1(adaptive).to(device)
    x = runner.bag(multiscale, largest, device)
    loss, _, _, _ = runner.step(probe, x, largest['label'], torch.nn.BCEWithLogitsLoss())
    loss.backward()
    torch.cuda.synchronize(device)
    peak = torch.cuda.max_memory_allocated(device)
    capacity = torch.cuda.get_device_properties(device).total_memory
    print(('adaptive' if adaptive else 'fixed'), 'GPU', gpu, 'slide', largest['slide_id'],
          'patches', largest['n_patches'], 'peak GiB', round(peak / 2**30, 2),
          'capacity GiB', round(capacity / 2**30, 2),
          'seconds', round(time.monotonic() - started, 2))
    assert peak < .8 * capacity, 'Large-bag pilot leaves insufficient GPU margin'
    del x, loss, probe
    torch.cuda.empty_cache()


In [ ]:
# One independent 15-fit worker per T4; automatically fall back to one GPU.
import os, subprocess, sys, time
from pathlib import Path

jobs = [('M-fixed', 'validate_fixed', 0), ('M-adaptive', 'validate_adaptive', 1)]
parallel = torch.cuda.device_count() >= 2
print('Parallel two-GPU validation:', parallel)
script = '/kaggle/working/multiscale_a1_runner.py'

def start_job(name, stage, gpu):
    destination = root / name
    if (destination / 'COMPLETE.json').is_file():
        runner.verify_variant(name, multiscale, destination)
        print('Verified completed output:', name)
        return None
    assert not destination.exists(), f'Partial output exists; inspect it before retrying: {destination}'
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    env['OMP_NUM_THREADS'] = env['MKL_NUM_THREADS'] = env['OPENBLAS_NUM_THREADS'] = '1'
    log = (root / (name + '.log')).open('w')
    process = subprocess.Popen([sys.executable, '-u', script, '--stage', stage,
        '--multi', str(multiscale), '--output', str(destination)],
        env=env, stdout=log, stderr=subprocess.STDOUT)
    return process, log

active = []
if parallel:
    for name, stage, gpu in jobs:
        job = start_job(name, stage, gpu)
        if job: active.append((name, *job))
    for name, process, log in active:
        returncode = process.wait()
        log.close()
        print(name, 'exit code', returncode, 'log:', root / (name + '.log'))
        if returncode:
            print((root / (name + '.log')).read_text()[-5000:])
    assert all(process.returncode == 0 for _, process, _ in active), 'Validation worker failed'
else:
    for name, stage, _ in jobs:
        job = start_job(name, stage, 0)
        if job:
            process, log = job
            returncode = process.wait()
            log.close()
            print(name, 'exit code', returncode)
            if returncode:
                print((root / (name + '.log')).read_text()[-5000:])
                raise RuntimeError(f'{name} validation failed')
for name, _, _ in jobs:
    runner.verify_variant(name, multiscale, root / name)
print('Both multiscale validation variants completed and verified.')


In [ ]:
# Final test begins only after both validation variants have completed.
import os, subprocess, sys, json
test_out = root / 'official-test'
assert not test_out.exists(), 'Test output already exists; inspect it before rerunning'
subprocess.run([sys.executable, '/kaggle/working/multiscale_a1_runner.py',
    '--stage', 'final_test', '--multi', str(multiscale), '--prior-a1', str(prior),
    '--fixed', str(root / 'M-fixed'), '--adaptive', str(root / 'M-adaptive'),
    '--output', str(test_out)], check=True)
complete = json.loads((test_out / 'COMPLETE.json').read_text())
assert complete['status'] == 'complete' and complete['models_per_variant'] == 15
import pandas as pd
display(pd.read_csv(test_out / 'test_comparison.csv'))
print('Save this completed notebook version with its /kaggle/working output.')
